# 阿瓦隆后续实验：thinking 预算、真实验证流水线、抽取修复（Qwen3-8B）

按顺序运行全部单元。说明见项目里的 `CODEX_FOLLOWUP.md`。

- 不克隆 avalonlogs，不重建题集：复用冻结的 510 题和第一次运行的 direct / extract / think 原始输出。
- 第 3 节需要上传 `avalon_followup_inputs.zip`（如果左侧文件面板里还没有）。
- 第 5 节（便宜，约 15 分钟）结束、第 6 节结束、全部结束时各打包一次；每次都尽量下载，自动下载失败就从文件面板手动下载。
- 模型输出逐批落盘。同一运行时内中断后，从第 2 节开始重新运行即可续跑，已完成的批次不会重算。

## 1 环境与 GPU

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver==4.16.0.0',
                       'transformers==4.57.6', 'safetensors>=0.4.3'])
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), '请选择支持 BF16 的 CUDA GPU'
MEM = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
print(torch.cuda.get_device_name(0), round(MEM, 1), 'GiB')
assert MEM >= 40, '显存不足 40 GiB：请换 A100 80GB / H100 / RTX PRO 6000 等大显存的卡'

## 2 写入代码

In [ ]:
import json, os, hashlib, zipfile, shutil
from pathlib import Path
ROOT = Path('/content/avalon_followup')
ROOT.mkdir(exist_ok=True)
ASSETS = json.loads('{"avalon/game.py": "\\"\\"\\"Vocabulary, tables and record format for The Resistance: Avalon.\\n\\nA *record* is the public history of one game (JSON-compatible dict):\\n\\n  {\\"n\\": 7,\\n   \\"config\\": {\\"merlin\\": 1, \\"percival\\": 1, \\"servant\\": 2, \\"morgana\\": 1, \\"minion\\": 2},\\n   \\"quests\\": [\\n      {\\"q\\": 1,\\n       \\"proposals\\": [{\\"leader\\": 0, \\"team\\": [2, 4], \\"approve\\": [0, 2, 3, 4], \\"approved\\": true}],\\n       \\"team\\": [2, 4], \\"fails\\": 0, \\"result\\": \\"success\\"},\\n      ...],\\n   \\"end\\": {\\"assassinated\\": 4, \\"outcome\\": \\"good_win\\"}}      # or null while the game runs\\n\\nPlayers are 0..n-1 in seating (clockwise) order; they are shown to people as 1..n.\\n`approve` lists the players who voted to approve (null if unknown).  `fails` is\\nthe number of Fail cards revealed (null if only the result is known).\\n\\nA *view* is the private information of one player:\\n  {\\"player\\": 3, \\"role\\": \\"percival\\", \\"seen\\": [0, 6], \\"lady\\": [[5, false]]}\\n`seen` is what the setup reveal showed that player (meaning depends on the\\nrole, see RULES.md K-*); `lady` lists Lady-of-the-Lake checks made by that player\\nas [target, target_is_evil].\\n\\n*Reveals* are extra alignment facts used by the information-completion\\nanalysis: [[player, is_evil], ...].\\n\\"\\"\\"\\nimport json\\n\\nROLES = (\'merlin\', \'percival\', \'servant\', \'morgana\', \'mordred\', \'oberon\', \'minion\')\\nGOOD_ROLES = (\'merlin\', \'percival\', \'servant\')\\nEVIL_ROLES = (\'morgana\', \'mordred\', \'oberon\', \'minion\')\\nSPECIAL = (\'merlin\', \'percival\', \'morgana\', \'mordred\', \'oberon\')\\n\\n# Rulebook tables (RULES.md S-COUNT, T-SIZE, Q-TWOFAIL)\\nEVIL_COUNT = {5: 2, 6: 2, 7: 3, 8: 3, 9: 3, 10: 4}\\nTEAM_SIZE = {5: (2, 3, 2, 3, 3), 6: (2, 3, 4, 3, 4), 7: (2, 3, 3, 4, 4),\\n             8: (3, 4, 4, 5, 5), 9: (3, 4, 4, 5, 5), 10: (3, 4, 4, 5, 5)}\\n\\n\\ndef fails_required(n, q):\\n    return 2 if q == 4 and n >= 7 else 1\\n\\n\\nZH_ROLE = {\'merlin\': \'梅林\', \'percival\': \'派西维尔\', \'servant\': \'亚瑟的忠臣\', \'morgana\': \'莫甘娜\',\\n           \'mordred\': \'莫德雷德\', \'oberon\': \'奥伯伦\', \'minion\': \'莫德雷德的爪牙\'}\\n\\nDEFAULT_VARIANT = {\\n    \'fail_count_public\': True,   # revealed Quest cards show how many Fails were played\\n    \'merlin_sees_oberon\': True,  # Oberon raises his thumb in the Merlin step\\n}\\n\\n\\ndef player_label(i):\\n    return f\'{i + 1}号\'\\n\\n\\ndef team_label(team):\\n    return \'、\'.join(player_label(i) for i in sorted(team))\\n\\n\\ndef normalize_role(text):\\n    t = str(text).lower()\\n    for key, role in ((\'merlin\', \'merlin\'), (\'percival\', \'percival\'), (\'morgana\', \'morgana\'),\\n                      (\'mordred\', \'mordred\'), (\'oberon\', \'oberon\'), (\'minion\', \'minion\'),\\n                      (\'assassin\', \'minion\'), (\'loyal\', \'servant\'), (\'servant\', \'servant\'),\\n                      (\'follower\', \'servant\'), (\'resistance\', \'servant\'), (\'spy\', \'minion\')):\\n        if key in t:\\n            return role\\n    raise ValueError(f\'unknown role {text!r}\')\\n\\n\\ndef config_ok(n, config):\\n    \\"\\"\\"Static setup check (S-COUNT, S-UNIQUE); returns list of violated rule ids.\\"\\"\\"\\n    bad = []\\n    if n not in EVIL_COUNT:\\n        bad.append(\'S-SIZE\')\\n        return bad\\n    if sum(config.values()) != n:\\n        bad.append(\'S-TOTAL\')\\n    if sum(config.get(r, 0) for r in EVIL_ROLES) != EVIL_COUNT[n]:\\n        bad.append(\'S-COUNT\')\\n    if any(config.get(r, 0) > 1 for r in SPECIAL):\\n        bad.append(\'S-UNIQUE\')\\n    if any(r not in ROLES for r in config):\\n        bad.append(\'S-ROLE\')\\n    return bad\\n\\n\\ndef dumps(x):\\n    return json.dumps(x, ensure_ascii=False, sort_keys=True)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Chinese rendering (used for test cases and LLM inputs)\\n# ---------------------------------------------------------------------------\\ndef render_config(n, config):\\n    parts = []\\n    for r in ROLES:\\n        c = config.get(r, 0)\\n        if c:\\n            parts.append(ZH_ROLE[r] + (f\'×{c}\' if c > 1 else \'\'))\\n    return f\'{n}人局，角色为：\' + \'、\'.join(parts) + f\'（共{EVIL_COUNT.get(n, \\"?\\")}名坏人）。\'\\n\\n\\ndef render_record(rec, show_votes=False):\\n    lines = [render_config(rec[\'n\'], rec[\'config\'])]\\n    for qd in rec.get(\'quests\', []):\\n        q = qd[\'q\']\\n        if show_votes:\\n            for k, p in enumerate(qd.get(\'proposals\', [])):\\n                s = f\'第{q}轮第{k + 1}次提案：队长{player_label(p[\\"leader\\"])}提名{team_label(p[\\"team\\"])}\'\\n                if p.get(\'approve\') is not None:\\n                    s += f\'，赞成者为{team_label(p[\\"approve\\"]) or \\"无\\"}\'\\n                s += \'，通过。\' if p[\'approved\'] else \'，被否决。\'\\n                lines.append(s)\\n        if qd.get(\'result\') is None:\\n            continue\\n        s = f\'第{q}个任务由{team_label(qd[\\"team\\"])}执行\'\\n        if qd.get(\'fails\') is not None:\\n            s += f\'，翻开{qd[\\"fails\\"]}张失败票\'\\n        s += \'，任务\' + (\'成功。\' if qd[\'result\'] == \'success\' else \'失败。\')\\n        lines.append(s)\\n    end = rec.get(\'end\')\\n    if end and end.get(\'assassinated\') is not None:\\n        lines.append(f\'三个任务成功后，刺客刺杀了{player_label(end[\\"assassinated\\"])}，\'\\n                     + (\'好人获胜。\' if end[\'outcome\'] == \'good_win\' else \'坏人获胜。\'))\\n    return \'\\\\n\'.join(lines)\\n\\n\\ndef render_view(view, config):\\n    if view is None:\\n        return \'你是旁观者，只知道公开信息。\'\\n    p, role = view[\'player\'], view[\'role\']\\n    s = f\'你是{player_label(p)}，身份是{ZH_ROLE[role]}。\'\\n    seen = view.get(\'seen\') or []\\n    if role == \'merlin\':\\n        s += f\'开局时你看到的坏人是：{team_label(seen) or \\"无人\\"}。\'\\n    elif role == \'percival\':\\n        s += f\'开局时你看到{team_label(seen)}\' + (\'（其中一人是梅林，另一人是莫甘娜，你分不清）。\' if len(seen) == 2 else \'是梅林。\')\\n    elif role in EVIL_ROLES and role != \'oberon\':\\n        s += f\'开局时你看到的同伴是：{team_label(seen) or \\"无人\\"}。\'\\n    for t, ev in view.get(\'lady\') or []:\\n        s += f\'你用湖中仙女查验了{player_label(t)}，其阵营为{\\"坏人\\" if ev else \\"好人\\"}。\'\\n    return s\\n", "avalon/smt.py": "\\"\\"\\"Z3 encoding of the Avalon rules (see RULES.md for the clause table).\\n\\nEvery rule instance is a tracked constraint whose tag starts with the clause\\nid (e.g. \'Q-GOOD@q3:5\' = \\"good players play Success\\", quest 3, seat 5), so\\nunsatisfiable cores read as lists of rule instances and recorded facts.\\nEach quest card is a Boolean variable (True = Fail); this is a different\\nformulation from oracle.py, which checks #Fail <= #evil on the team directly.\\n\\"\\"\\"\\nimport z3\\nfrom game import (ROLES, EVIL_ROLES, SPECIAL, EVIL_COUNT, TEAM_SIZE, DEFAULT_VARIANT,\\n                  fails_required)\\n\\nCODE = {r: i for i, r in enumerate(ROLES)}\\n\\n\\nclass SolverUnknown(RuntimeError):\\n    pass\\n\\n\\ndef _g(b):\\n    return z3.BoolVal(bool(b))\\n\\n\\nclass SMT:\\n    name = \'z3\'\\n\\n    def __init__(self, timeout_ms=60000):\\n        self.timeout = timeout_ms\\n        self.calls = 0\\n\\n    # ------------------------------------------------------------------ encoding\\n    def encode(self, inst, suffix=\'\'):\\n        rec, view = inst[\'rec\'], inst.get(\'view\')\\n        reveals = inst.get(\'reveals\') or []\\n        variant = {**DEFAULT_VARIANT, \'check_leader\': True, **inst.get(\'variant\', {})}\\n        n, cfg = rec[\'n\'], rec[\'config\']\\n        R = [z3.Int(f\'role{suffix}_{i}\') for i in range(n)]\\n        E = [z3.Or([x == CODE[r] for r in EVIL_ROLES]) for x in R]\\n        bg = [z3.And(x >= 0, x < len(ROLES)) for x in R]\\n        tr = []\\n\\n        def add(tag, expr):\\n            tr.append((tag, expr))\\n\\n        add(\'S-SIZE\', _g(n in EVIL_COUNT))\\n        if n not in EVIL_COUNT:\\n            return bg, tr, R, E\\n        add(\'S-TOTAL\', _g(sum(cfg.values()) == n))\\n        add(\'S-COUNT\', _g(sum(cfg.get(r, 0) for r in EVIL_ROLES) == EVIL_COUNT[n]))\\n        add(\'S-UNIQUE\', _g(all(cfg.get(r, 0) <= 1 for r in SPECIAL)))\\n        for r in ROLES:\\n            add(f\'S-DEAL[{r}]\', z3.Sum([z3.If(x == CODE[r], 1, 0) for x in R]) == cfg.get(r, 0))\\n\\n        quests = rec.get(\'quests\', [])\\n        end = rec.get(\'end\')\\n        wins = losses = 0\\n        last_leader = None\\n        for pos, qd in enumerate(quests):\\n            q = qd[\'q\']\\n            add(f\'R-ORDER@{pos + 1}\', _g(q == pos + 1))\\n            if q != pos + 1:\\n                continue\\n            add(f\'G-END@q{q}\', _g(wins < 3 and losses < 3))\\n            size = TEAM_SIZE[n][q - 1]\\n            props = qd.get(\'proposals\', [])\\n            add(f\'T-HAMMER@q{q}\', _g(len(props) <= 5))\\n            for k, p in enumerate(props):\\n                tag = f\'q{q}.{k + 1}\'\\n                team = p[\'team\']\\n                add(f\'T-SIZE@{tag}\', _g(len(team) == size))\\n                add(f\'T-DISTINCT@{tag}\', _g(len(set(team)) == len(team) and all(0 <= x < n for x in team)))\\n                if p.get(\'approve\') is not None:\\n                    add(f\'T-VOTE@{tag}\', _g((2 * len(set(p[\'approve\'])) > n) == bool(p[\'approved\'])))\\n                if variant[\'check_leader\'] and last_leader is not None:\\n                    add(f\'T-LEADER@{tag}\', _g(p[\'leader\'] == (last_leader + 1) % n))\\n                last_leader = p[\'leader\']\\n                if p[\'approved\'] and k != len(props) - 1:\\n                    add(f\'T-LAST@{tag}\', _g(False))\\n            if qd.get(\'result\') is not None:\\n                team = qd[\'team\']\\n                if props:\\n                    add(f\'T-LAST@q{q}\', _g(props[-1][\'approved\'] and sorted(props[-1][\'team\']) == sorted(team)))\\n                add(f\'Q-SIZE@q{q}\', _g(len(team) == size and len(set(team)) == size))\\n                cards = {i: z3.Bool(f\'card{suffix}_{q}_{i}\') for i in sorted(set(team)) if 0 <= i < n}\\n                for i, c in cards.items():\\n                    add(f\'Q-GOOD@q{q}:{i}\', z3.Implies(z3.Not(E[i]), z3.Not(c)))\\n                nf = z3.Sum([z3.If(c, 1, 0) for c in cards.values()]) if cards else z3.IntVal(0)\\n                f, need, failed = qd.get(\'fails\'), fails_required(n, q), qd[\'result\'] == \'fail\'\\n                if variant[\'fail_count_public\'] and f is not None:\\n                    add(f\'Q-COUNT@q{q}\', nf == f)\\n                    add(f\'Q-RESULT@q{q}\', _g((f >= need) == failed))\\n                else:\\n                    add(f\'Q-RESULT@q{q}\', (nf >= need) == failed)\\n                if failed:\\n                    losses += 1\\n                else:\\n                    wins += 1\\n            elif len(props) == 5 and not props[-1][\'approved\']:\\n                ok = pos == len(quests) - 1 and not (end and end.get(\'outcome\') not in (None, \'evil_win\'))\\n                add(f\'T-HAMMER@q{q}:end\', _g(ok))\\n\\n        resolved = {qd[\'q\'] for qd in quests if qd.get(\'result\') is not None}\\n        holders, prev_q = [], 0\\n        for k, use in enumerate(rec.get(\'lady\') or []):\\n            add(f\'L-WHEN@{k + 1}\', _g(use[\'q\'] in (2, 3, 4) and use[\'q\'] > prev_q and use[\'q\'] in resolved))\\n            prev_q = use[\'q\']\\n            if k == 0:\\n                first = quests[0][\'proposals\'][0][\'leader\'] if quests and quests[0].get(\'proposals\') else None\\n                if first is not None:\\n                    add(\'L-FIRST\', _g(use[\'holder\'] == (first - 1) % n))\\n            else:\\n                add(f\'L-CHAIN@{k + 1}\', _g(use[\'holder\'] == rec[\'lady\'][k - 1][\'target\']))\\n            holders.append(use[\'holder\'])\\n            add(f\'L-PREV@{k + 1}\', _g(use[\'target\'] not in holders and 0 <= use[\'target\'] < n))\\n\\n        if end and end.get(\'outcome\') is not None:\\n            add(\'G-OUTCOME\', _g(not (losses >= 3 and end[\'outcome\'] != \'evil_win\')))\\n            a = end.get(\'assassinated\')\\n            if a is not None:\\n                add(\'G-ASSASSIN\', _g(wins >= 3))\\n                if 0 <= a < n:\\n                    add(\'G-MERLIN-HIT\', (R[a] == CODE[\'merlin\']) == _g(end[\'outcome\'] == \'evil_win\'))\\n\\n        if view is not None:\\n            p, r = view[\'player\'], view[\'role\']\\n            seen = set(view.get(\'seen\') or [])\\n            add(\'V-SELF\', R[p] == CODE[r])\\n            add(\'K-SELF\', _g(p not in seen and all(0 <= j < n for j in seen)))\\n            for j in range(n):\\n                if j == p:\\n                    continue\\n                inS = _g(j in seen)\\n                if r == \'merlin\':\\n                    vis = z3.And(E[j], R[j] != CODE[\'mordred\'])\\n                    if not variant[\'merlin_sees_oberon\']:\\n                        vis = z3.And(vis, R[j] != CODE[\'oberon\'])\\n                    add(f\'K-MERLIN@{j}\', vis == inS)\\n                elif r == \'percival\':\\n                    add(f\'K-PERCIVAL@{j}\', z3.Or(R[j] == CODE[\'merlin\'], R[j] == CODE[\'morgana\']) == inS)\\n                elif r in EVIL_ROLES and r != \'oberon\':\\n                    add(f\'K-EVIL@{j}\', z3.And(E[j], R[j] != CODE[\'oberon\']) == inS)\\n                else:\\n                    add(f\'K-NONE@{j}\', _g(j not in seen))\\n            for t, ev in view.get(\'lady\') or []:\\n                add(f\'K-LADY@{t}\', E[t] == _g(ev))\\n        for t, ev in reveals:\\n            add(f\'X-REVEAL@{t}\', E[t] == _g(ev))\\n        return bg, tr, R, E\\n\\n    def claim(self, c, R, E, rec):\\n        t = c[\'t\']\\n        cnt = lambda ps: z3.Sum([z3.If(E[i], 1, 0) for i in ps]) if ps else z3.IntVal(0)\\n        if t == \'evil\':\\n            return E[c[\'p\']]\\n        if t == \'good\':\\n            return z3.Not(E[c[\'p\']])\\n        if t == \'role\':\\n            return R[c[\'p\']] == CODE[c[\'r\']]\\n        if t == \'atleast\':\\n            return cnt(c[\'ps\']) >= c[\'k\']\\n        if t == \'exactly\':\\n            return cnt(c[\'ps\']) == c[\'k\']\\n        if t == \'same\':\\n            a, b = c[\'ps\']\\n            return E[a] == E[b]\\n        if t == \'evil_set\':\\n            S = set(c[\'ps\'])\\n            return z3.And([E[i] == _g(i in S) for i in range(len(R))])\\n        if t == \'not\':\\n            return z3.Not(self.claim(c[\'a\'], R, E, rec))\\n        if t == \'and\':\\n            return z3.And([self.claim(x, R, E, rec) for x in c[\'as\']])\\n        if t == \'or\':\\n            return z3.Or([self.claim(x, R, E, rec) for x in c[\'as\']])\\n        if t == \'fail_by\':\\n            return _g(c[\'f\'] >= fails_required(rec[\'n\'], c[\'q\']))\\n        if t == \'vote_pass\':\\n            return _g(2 * c[\'yes\'] > rec[\'n\'])\\n        if t == \'team_size\':\\n            return _g(TEAM_SIZE[rec[\'n\']][c[\'q\'] - 1] == c[\'k\'])\\n        raise ValueError(t)\\n\\n    # ------------------------------------------------------------------ solving\\n    def _solver(self):\\n        s = z3.Solver()\\n        s.set(timeout=self.timeout)\\n        return s\\n\\n    def _check(self, s, *assume):\\n        self.calls += 1\\n        r = s.check(*assume)\\n        if r == z3.unknown:\\n            raise SolverUnknown(s.reason_unknown())\\n        return r\\n\\n    @staticmethod\\n    def _decode(m, R):\\n        return tuple(ROLES[m.eval(x, model_completion=True).as_long()] for x in R)\\n\\n    def classify(self, inst, claim, explain=False):\\n        bg, tr, R, E = self.encode(inst)\\n        s = self._solver()\\n        s.add(*bg)\\n        s.add(*[e for _, e in tr])\\n        if self._check(s) == z3.unsat:\\n            out = {\'label\': \'inconsistent\', \'positive\': None, \'negative\': None}\\n            if explain:\\n                out[\'core\'] = self.core(bg, tr)\\n            return out\\n        cz = self.claim(claim, R, E, inst[\'rec\'])\\n        wit = []\\n        for side in (cz, z3.Not(cz)):\\n            s.push()\\n            s.add(side)\\n            wit.append(self._decode(s.model(), R) if self._check(s) == z3.sat else None)\\n            s.pop()\\n        pos, neg = wit\\n        label = \'undetermined\' if pos and neg else (\'entailed\' if pos else \'contradicted\')\\n        out = {\'label\': label, \'positive\': pos, \'negative\': neg}\\n        if explain and label == \'entailed\':\\n            out[\'core\'] = self.core(bg, tr, z3.Not(cz))\\n        if explain and label == \'contradicted\':\\n            out[\'core\'] = self.core(bg, tr, cz)\\n        return out\\n\\n    def core(self, bg, tr, extra=None):\\n        \\"\\"\\"Subset-minimal set of tracked rule instances (deletion-based).\\"\\"\\"\\n        s = self._solver()\\n        s.add(*bg)\\n        if extra is not None:\\n            s.add(extra)\\n        lits = [z3.Bool(f\'__t{i}\') for i in range(len(tr))]\\n        for lit, (_, e) in zip(lits, tr):\\n            s.add(z3.Implies(lit, e))\\n        if self._check(s, *lits) != z3.unsat:\\n            raise AssertionError(\'core of a satisfiable query\')\\n        index = {str(l): i for i, l in enumerate(lits)}\\n        cur = sorted(index[str(x)] for x in s.unsat_core())\\n        i = 0\\n        while i < len(cur):\\n            trial = cur[:i] + cur[i + 1:]\\n            if self._check(s, *[lits[j] for j in trial]) == z3.unsat:\\n                cur = trial\\n            else:\\n                i += 1\\n        return [tr[j][0] for j in cur]\\n\\n    def find_pair(self, inst, claim, agree):\\n        bga, tra, Ra, Ea = self.encode(inst, \'_a\')\\n        bgb, trb, Rb, Eb = self.encode(inst, \'_b\')\\n        s = self._solver()\\n        s.add(*bga, *bgb, *[e for _, e in tra], *[e for _, e in trb])\\n        s.add(self.claim(claim, Ra, Ea, inst[\'rec\']) != self.claim(claim, Rb, Eb, inst[\'rec\']))\\n        s.add(*[Ea[j] == Eb[j] for j in agree])\\n        if self._check(s) == z3.unsat:\\n            return None\\n        m = s.model()\\n        return self._decode(m, Ra), self._decode(m, Rb)\\n\\n    def determined_players(self, inst):\\n        \\"\\"\\"For each seat: \'evil\' / \'good\' / \'unknown\' (or None if inconsistent).\\"\\"\\"\\n        bg, tr, R, E = self.encode(inst)\\n        s = self._solver()\\n        s.add(*bg, *[e for _, e in tr])\\n        if self._check(s) == z3.unsat:\\n            return None\\n        out = []\\n        for e in E:\\n            s.push(); s.add(e); can_evil = self._check(s) == z3.sat; s.pop()\\n            s.push(); s.add(z3.Not(e)); can_good = self._check(s) == z3.sat; s.pop()\\n            out.append(\'unknown\' if can_evil and can_good else (\'evil\' if can_evil else \'good\'))\\n        return out\\n\\n    def possible_roles(self, inst, p):\\n        bg, tr, R, E = self.encode(inst)\\n        s = self._solver()\\n        s.add(*bg, *[e for _, e in tr])\\n        out = []\\n        for r in ROLES:\\n            s.push()\\n            s.add(R[p] == CODE[r])\\n            if self._check(s) == z3.sat:\\n                out.append(r)\\n            s.pop()\\n        return out\\n", "avalon/search.py": "\\"\\"\\"Minimal information completion for Avalon claims (backend-agnostic).\\n\\nA *reveal* tells the alignment (good/evil) of one player, exactly what a Lady\\nof the Lake check gives its holder.  For an undetermined claim C we ask:\\n\\n  one-shot   : the smallest set S of players whose alignments, whatever they\\n               turn out to be, always decide C.  S is sufficient iff no two\\n               consistent worlds agree on the alignments in S but disagree on C\\n               (two-world test, backend.find_pair).\\n               - naive : enumerate subsets by increasing size\\n               - cegar : counterexample-guided implicit hitting sets\\n  adaptive   : the minimax number of sequential reveals, where each next\\n               player is chosen after seeing earlier answers (a game against\\n               an adversarial world; answers must stay consistent).\\n\\nBackends implement classify(inst, claim) and find_pair(inst, claim, agree).\\n\\"\\"\\"\\nimport time\\nfrom itertools import combinations\\n\\nEVIL = {\'morgana\', \'mordred\', \'oberon\', \'minion\'}\\nDECIDED = (\'entailed\', \'contradicted\')\\n\\n\\ndef _ev(roles, j):\\n    return roles[j] in EVIL\\n\\n\\ndef min_hitting_set(sets, costs):\\n    best = [float(\'inf\'), None]\\n    sets = [frozenset(s) for s in sets]\\n    if any(not s for s in sets):\\n        return None\\n\\n    def rec(chosen, cost):\\n        if cost >= best[0]:\\n            return\\n        unhit = None\\n        for s in sets:\\n            if not (s & chosen) and (unhit is None or len(s) < len(unhit)):\\n                unhit = s\\n        if unhit is None:\\n            best[0], best[1] = cost, chosen\\n            return\\n        for f in sorted(unhit, key=lambda x: (costs[x], x)):\\n            rec(chosen | {f}, cost + costs[f])\\n    rec(frozenset(), 0)\\n    return None if best[1] is None else sorted(best[1])\\n\\n\\ndef candidates(inst):\\n    known = {t for t, _ in inst.get(\'reveals\') or []}\\n    view = inst.get(\'view\')\\n    if view is not None:\\n        known.add(view[\'player\'])\\n        known |= {t for t, _ in view.get(\'lady\') or []}\\n    return [j for j in range(inst[\'rec\'][\'n\']) if j not in known]\\n\\n\\ndef min_reveal_naive(backend, inst, claim, cands=None, deadline=None):\\n    cands = candidates(inst) if cands is None else cands\\n    checks, t0 = 0, time.perf_counter()\\n    for k in range(len(cands) + 1):\\n        found = []\\n        for S in combinations(cands, k):\\n            if deadline and time.perf_counter() - t0 > deadline:\\n                return {\'status\': \'timeout\', \'checks\': checks, \'size_reached\': k}\\n            checks += 1\\n            if backend.find_pair(inst, claim, list(S)) is None:\\n                found.append(list(S))\\n        if found:\\n            return {\'status\': \'ok\', \'sets\': found, \'size\': k, \'checks\': checks}\\n    return {\'status\': \'none\', \'checks\': checks}\\n\\n\\ndef min_reveal_cegar(backend, inst, claim, cands=None, costs=None, generalize=True):\\n    cands = candidates(inst) if cands is None else cands\\n    costs = costs or {j: 1 for j in cands}\\n    D, calls, rounds = [], 0, 0\\n    while True:\\n        rounds += 1\\n        H = min_hitting_set(D, costs)\\n        if H is None:\\n            return {\'status\': \'none\', \'calls\': calls, \'rounds\': rounds}\\n        pair = backend.find_pair(inst, claim, H)\\n        calls += 1\\n        if pair is None:\\n            return {\'status\': \'ok\', \'set\': H, \'size\': len(H), \'calls\': calls, \'rounds\': rounds,\\n                    \'cuts\': [sorted(d) for d in D]}\\n        x, y = pair\\n        diff = [j for j in cands if j not in H and _ev(x, j) != _ev(y, j)]\\n        if generalize:\\n            agree = list(H)\\n            for j in list(diff):\\n                if j not in diff:\\n                    continue\\n                trial = backend.find_pair(inst, claim, agree + [j])\\n                calls += 1\\n                if trial is not None:\\n                    agree.append(j)\\n                    x, y = trial\\n                    diff = [g for g in cands if g not in agree and _ev(x, g) != _ev(y, g)]\\n        if not diff:\\n            return {\'status\': \'none\', \'calls\': calls, \'rounds\': rounds}\\n        D.append(frozenset(diff))\\n\\n\\ndef adaptive_depth(backend, inst, claim, cands=None, limit=None):\\n    \\"\\"\\"Worst-case number of sequential reveals needed to decide the claim.\\n\\n    Returns (depth, first_choice); depth is None if no reveal strategy over\\n    `cands` can decide the claim.\\"\\"\\"\\n    cands = candidates(inst) if cands is None else cands\\n    memo = {}\\n    base = list(inst.get(\'reveals\') or [])\\n\\n    def label(extra):\\n        return backend.classify({**inst, \'reveals\': base + list(extra)}, claim)[\'label\']\\n\\n    def solve(extra):\\n        key = frozenset(extra)\\n        if key in memo:\\n            return memo[key]\\n        lab = label(extra)\\n        if lab in DECIDED or lab == \'inconsistent\':\\n            memo[key] = (0, None)\\n            return memo[key]\\n        used = {t for t, _ in extra}\\n        best = (None, None)\\n        for j in cands:\\n            if j in used:\\n                continue\\n            worst = 0\\n            for v in (False, True):\\n                nxt = tuple(sorted(extra + ((j, v),)))\\n                if label(nxt) == \'inconsistent\':\\n                    continue\\n                d, _ = solve(nxt)\\n                if d is None:\\n                    worst = None\\n                    break\\n                worst = max(worst, d + 1)\\n            if worst is not None and (best[0] is None or worst < best[0]):\\n                best = (worst, j)\\n                if limit is not None and worst <= 1:\\n                    break\\n        memo[key] = best\\n        return best\\n\\n    return solve(tuple())\\n", "avalon/oracle.py": "\\"\\"\\"Independent executable reading of the Avalon rules (no solver).\\n\\nEnumerates every assignment of the configured role multiset to the seats and\\nkeeps those consistent with the record, the private view and the reveals.\\nIt shares no rule code with smt.py: the tables are typed again here, the\\nquest rule is checked as an inequality (#Fail cards <= #evil on the team)\\ninstead of per-card variables, and record legality is checked imperatively.\\nselfcheck.py compares the two implementations instance by instance.\\n\\"\\"\\"\\nfrom itertools import combinations\\n\\nO_EVIL = {\'morgana\', \'mordred\', \'oberon\', \'minion\'}\\nO_ORDER = (\'merlin\', \'percival\', \'servant\', \'morgana\', \'mordred\', \'oberon\', \'minion\')\\nO_EVIL_COUNT = {5: 2, 6: 2, 7: 3, 8: 3, 9: 3, 10: 4}\\nO_SIZES = {5: [2, 3, 2, 3, 3], 6: [2, 3, 4, 3, 4], 7: [2, 3, 3, 4, 4],\\n           8: [3, 4, 4, 5, 5], 9: [3, 4, 4, 5, 5], 10: [3, 4, 4, 5, 5]}\\n\\n\\ndef o_need(n, q):\\n    if q == 4 and n >= 7:\\n        return 2\\n    return 1\\n\\n\\ndef role_assignments(n, config):\\n    left = {r: config.get(r, 0) for r in O_ORDER}\\n    seats = [None] * n\\n\\n    def go(i):\\n        if i == n:\\n            yield tuple(seats)\\n            return\\n        for r in O_ORDER:\\n            if left[r]:\\n                left[r] -= 1\\n                seats[i] = r\\n                yield from go(i + 1)\\n                left[r] += 1\\n    if sum(left.values()) == n:\\n        yield from go(0)\\n\\n\\ndef record_violations(rec, variant):\\n    \\"\\"\\"Role-independent legality of the record; returns violated rule ids.\\"\\"\\"\\n    n, cfg, bad = rec[\'n\'], rec[\'config\'], []\\n    if n not in O_EVIL_COUNT:\\n        return [\'S-SIZE\']\\n    if sum(cfg.values()) != n:\\n        bad.append(\'S-TOTAL\')\\n    if sum(v for r, v in cfg.items() if r in O_EVIL) != O_EVIL_COUNT[n]:\\n        bad.append(\'S-COUNT\')\\n    if any(cfg.get(r, 0) > 1 for r in (\'merlin\', \'percival\', \'morgana\', \'mordred\', \'oberon\')):\\n        bad.append(\'S-UNIQUE\')\\n    wins = losses = 0\\n    last_leader = None\\n    quests = rec.get(\'quests\', [])\\n    for pos, qd in enumerate(quests):\\n        q = qd[\'q\']\\n        if q != pos + 1:\\n            bad.append(\'R-ORDER\')\\n            continue\\n        if wins >= 3 or losses >= 3:\\n            bad.append(\'G-END\')\\n        size = O_SIZES[n][q - 1]\\n        props = qd.get(\'proposals\', [])\\n        if len(props) > 5:\\n            bad.append(\'T-HAMMER\')\\n        for k, p in enumerate(props):\\n            team = p[\'team\']\\n            if len(team) != size:\\n                bad.append(\'T-SIZE\')\\n            if len(set(team)) != len(team) or any(not 0 <= x < n for x in team):\\n                bad.append(\'T-DISTINCT\')\\n            if p.get(\'approve\') is not None:\\n                yes = len(set(p[\'approve\']))\\n                if (2 * yes > n) != bool(p[\'approved\']):\\n                    bad.append(\'T-VOTE\')\\n            if variant.get(\'check_leader\', True) and last_leader is not None:\\n                if p[\'leader\'] != (last_leader + 1) % n:\\n                    bad.append(\'T-LEADER\')\\n            last_leader = p[\'leader\']\\n            if p[\'approved\'] and k != len(props) - 1:\\n                bad.append(\'T-LAST\')\\n        if qd.get(\'result\') is not None:\\n            if not props or not props[-1][\'approved\'] or sorted(props[-1][\'team\']) != sorted(qd[\'team\']):\\n                if props:\\n                    bad.append(\'T-LAST\')\\n            if len(qd[\'team\']) != size or len(set(qd[\'team\'])) != size:\\n                bad.append(\'Q-SIZE\')\\n            f = qd.get(\'fails\')\\n            if variant.get(\'fail_count_public\', True) and f is not None:\\n                if f < 0 or f > len(qd[\'team\']):\\n                    bad.append(\'Q-COUNT\')\\n                if (f >= o_need(n, q)) != (qd[\'result\'] == \'fail\'):\\n                    bad.append(\'Q-RESULT\')\\n            if qd[\'result\'] == \'success\':\\n                wins += 1\\n            else:\\n                losses += 1\\n        elif len(props) == 5 and not props[-1][\'approved\']:\\n            # fifth rejection: game over, evil wins (T-HAMMER)\\n            if pos != len(quests) - 1:\\n                bad.append(\'T-HAMMER\')\\n            end = rec.get(\'end\')\\n            if end and end.get(\'outcome\') not in (None, \'evil_win\'):\\n                bad.append(\'T-HAMMER\')\\n    # Lady of the Lake (optional rule): public part of each use\\n    resolved = {qd[\'q\'] for qd in quests if qd.get(\'result\') is not None}\\n    holders, prev_q = [], 0\\n    for k, use in enumerate(rec.get(\'lady\') or []):\\n        if use[\'q\'] not in (2, 3, 4) or use[\'q\'] <= prev_q or use[\'q\'] not in resolved:\\n            bad.append(\'L-WHEN\')\\n        prev_q = use[\'q\']\\n        if k == 0:\\n            first = quests[0][\'proposals\'][0][\'leader\'] if quests and quests[0].get(\'proposals\') else None\\n            if first is not None and use[\'holder\'] != (first - 1) % n:\\n                bad.append(\'L-FIRST\')\\n        elif use[\'holder\'] != rec[\'lady\'][k - 1][\'target\']:\\n            bad.append(\'L-CHAIN\')\\n        holders.append(use[\'holder\'])\\n        if use[\'target\'] in holders or not 0 <= use[\'target\'] < n:\\n            bad.append(\'L-PREV\')\\n    end = rec.get(\'end\')\\n    if end and end.get(\'outcome\') is not None:\\n        if losses >= 3 and end[\'outcome\'] != \'evil_win\':\\n            bad.append(\'G-OUTCOME\')\\n        if end.get(\'assassinated\') is not None and wins < 3:\\n            bad.append(\'G-ASSASSIN\')\\n    return sorted(set(bad))\\n\\n\\ndef _team_evil(roles, team):\\n    return sum(1 for i in team if roles[i] in O_EVIL)\\n\\n\\ndef world_ok(roles, rec, view, reveals, variant):\\n    n = rec[\'n\']\\n    for qd in rec.get(\'quests\', []):\\n        if qd.get(\'result\') is None:\\n            continue\\n        ev = _team_evil(roles, qd[\'team\'])\\n        f = qd.get(\'fails\')\\n        if variant.get(\'fail_count_public\', True) and f is not None:\\n            if f > ev:\\n                return False\\n        elif qd[\'result\'] == \'fail\' and ev < o_need(rec[\'n\'], qd[\'q\']):\\n            return False\\n    end = rec.get(\'end\')\\n    if end and end.get(\'assassinated\') is not None:\\n        hit = roles[end[\'assassinated\']] == \'merlin\'\\n        if (end[\'outcome\'] == \'evil_win\') != hit:\\n            return False\\n    if view is not None:\\n        p, r = view[\'player\'], view[\'role\']\\n        if roles[p] != r:\\n            return False\\n        seen = set(view.get(\'seen\') or [])\\n        if r == \'merlin\':\\n            exp = set()\\n            for j in range(n):\\n                if j == p or roles[j] not in O_EVIL or roles[j] == \'mordred\':\\n                    continue\\n                if roles[j] == \'oberon\' and not variant.get(\'merlin_sees_oberon\', True):\\n                    continue\\n                exp.add(j)\\n        elif r == \'percival\':\\n            exp = {j for j in range(n) if j != p and roles[j] in (\'merlin\', \'morgana\')}\\n        elif r in O_EVIL and r != \'oberon\':\\n            exp = {j for j in range(n) if j != p and roles[j] in O_EVIL and roles[j] != \'oberon\'}\\n        else:\\n            exp = set()  # servants and Oberon learn nothing at setup\\n        if exp != seen:\\n            return False\\n        for t, is_evil in view.get(\'lady\') or []:\\n            if (roles[t] in O_EVIL) != bool(is_evil):\\n                return False\\n    for t, is_evil in reveals or []:\\n        if (roles[t] in O_EVIL) != bool(is_evil):\\n            return False\\n    return True\\n\\n\\ndef claim_holds(c, roles, rec):\\n    t = c[\'t\']\\n    if t == \'evil\':\\n        return roles[c[\'p\']] in O_EVIL\\n    if t == \'good\':\\n        return roles[c[\'p\']] not in O_EVIL\\n    if t == \'role\':\\n        return roles[c[\'p\']] == c[\'r\']\\n    if t == \'atleast\':\\n        return _team_evil(roles, c[\'ps\']) >= c[\'k\']\\n    if t == \'exactly\':\\n        return _team_evil(roles, c[\'ps\']) == c[\'k\']\\n    if t == \'same\':\\n        a, b = c[\'ps\']\\n        return (roles[a] in O_EVIL) == (roles[b] in O_EVIL)\\n    if t == \'evil_set\':\\n        return {i for i, r in enumerate(roles) if r in O_EVIL} == set(c[\'ps\'])\\n    if t == \'not\':\\n        return not claim_holds(c[\'a\'], roles, rec)\\n    if t == \'and\':\\n        return all(claim_holds(x, roles, rec) for x in c[\'as\'])\\n    if t == \'or\':\\n        return any(claim_holds(x, roles, rec) for x in c[\'as\'])\\n    if t == \'fail_by\':\\n        return c[\'f\'] >= o_need(rec[\'n\'], c[\'q\'])\\n    if t == \'vote_pass\':\\n        return 2 * c[\'yes\'] > rec[\'n\']\\n    if t == \'team_size\':\\n        return O_SIZES[rec[\'n\']][c[\'q\'] - 1] == c[\'k\']\\n    raise ValueError(t)\\n\\n\\ndef worlds(inst):\\n    rec, view = inst[\'rec\'], inst.get(\'view\')\\n    variant = inst.get(\'variant\', {})\\n    if record_violations(rec, variant):\\n        return\\n    for roles in role_assignments(rec[\'n\'], rec[\'config\']):\\n        if world_ok(roles, rec, view, inst.get(\'reveals\'), variant):\\n            yield roles\\n\\n\\nclass Oracle:\\n    name = \'oracle\'\\n\\n    def classify(self, inst, claim):\\n        pos = neg = None\\n        for roles in worlds(inst):\\n            if claim_holds(claim, roles, inst[\'rec\']):\\n                pos = pos or roles\\n            else:\\n                neg = neg or roles\\n            if pos and neg:\\n                break\\n        if pos is None and neg is None:\\n            return {\'label\': \'inconsistent\', \'positive\': None, \'negative\': None}\\n        label = \'undetermined\' if pos and neg else (\'entailed\' if pos else \'contradicted\')\\n        return {\'label\': label, \'positive\': pos, \'negative\': neg}\\n\\n    def find_pair(self, inst, claim, agree):\\n        seen = {}\\n        for roles in worlds(inst):\\n            key = tuple(roles[j] in O_EVIL for j in agree)\\n            v = claim_holds(claim, roles, inst[\'rec\'])\\n            bucket = seen.setdefault(key, {})\\n            if (not v) in bucket:\\n                return bucket[not v], roles\\n            bucket.setdefault(v, roles)\\n        return None\\n\\n    def min_reveal_sets(self, inst, claim, candidates):\\n        ws = [(r, claim_holds(claim, r, inst[\'rec\'])) for r in worlds(inst)]\\n        if not ws:\\n            return None\\n        for k in range(len(candidates) + 1):\\n            found = []\\n            for S in combinations(candidates, k):\\n                groups = {}\\n                for r, v in ws:\\n                    groups.setdefault(tuple(r[j] in O_EVIL for j in S), set()).add(v)\\n                if all(len(g) == 1 for g in groups.values()):\\n                    found.append(list(S))\\n            if found:\\n                return found\\n        return []\\n\\n    def count_worlds(self, inst):\\n        return sum(1 for _ in worlds(inst))\\n\\n\\ndef determined_players(inst):\\n    \\"\\"\\"Per seat: \'evil\' / \'good\' / \'unknown\' over all consistent worlds (None if inconsistent).\\"\\"\\"\\n    seen = None\\n    for roles in worlds(inst):\\n        ev = [r in O_EVIL for r in roles]\\n        seen = [{e} for e in ev] if seen is None else [s | {e} for s, e in zip(seen, ev)]\\n    if seen is None:\\n        return None\\n    return [\'unknown\' if len(s) == 2 else (\'evil\' if True in s else \'good\') for s in seen]\\n\\n\\ndef possible_roles(inst, p):\\n    return sorted({roles[p] for roles in worlds(inst)})\\n", "avalon/llm_run.py": "\\"\\"\\"LLM experiment on frozen Avalon tasks (runs on a Colab A100).\\n\\nConditions (same items, same rule card):\\n  direct    : Qwen3-8B, thinking off, greedy, one-word verdict\\n  think     : Qwen3-8B, thinking on (sampling as recommended for thinking mode)\\n  extract   : the model only translates the text into the formal JSON; Z3 decides\\n  revise_*  : wrong direct answers get solver feedback and one more turn\\n              flag     = \\"your answer is not supported\\" only\\n              evidence = plus two consistent worlds or the minimal conflicting rules\\n  reveal    : for undetermined items, the model proposes Lady-of-the-Lake checks;\\n              scored for sufficiency (two-world test) and minimality\\n\\"\\"\\"\\nimport hashlib\\nimport json\\nimport random\\nimport re\\nimport time\\nfrom collections import Counter, defaultdict\\nfrom pathlib import Path\\n\\nfrom game import ZH_ROLE, ROLES, EVIL_ROLES, TEAM_SIZE, fails_required, player_label, team_label\\n\\nMODEL = \'Qwen/Qwen3-8B\'\\nREVISION = \'b968826d9c46dd6066d109eabc6255188de91218\'\\nLABELS = (\'entailed\', \'contradicted\', \'undetermined\', \'inconsistent\')\\nWORD = {\'entailed\': \'成立\', \'contradicted\': \'不成立\', \'undetermined\': \'不确定\', \'inconsistent\': \'矛盾\'}\\nFROM_WORD = {v: k for k, v in WORD.items()}\\n\\nRULE_CARD = \'\'\'你在分析一局《阿瓦隆》（The Resistance: Avalon）。只能依据下列规则和给出的记录做严格的逻辑推理；发言、投票倾向、\\"像不像\\"之类的策略性推测都不算证据。\\n\\n【阵营与人数】好人阵营：梅林、派西维尔、亚瑟的忠臣。坏人阵营：莫甘娜、莫德雷德、奥伯伦、莫德雷德的爪牙。坏人人数：5人局2名，6人局2名，7人局3名，8人局3名，9人局3名，10人局4名。本局使用的角色在记录开头给出，每人一张。\\n【开局信息】每个人知道自己的角色。除奥伯伦外的坏人互相认识，他们看不到奥伯伦，奥伯伦也看不到他们。梅林能看到除莫德雷德以外的所有坏人（包括奥伯伦）。派西维尔能看到梅林和莫甘娜，但分不清谁是谁。亚瑟的忠臣和奥伯伦开局得不到任何信息。\\n【组队与投票】队长提名队伍，所有人公开投票，赞成票严格过半才通过（平票算否决）。每次提案后队长按座位顺序交给下一位。同一轮连续5次否决则坏人直接获胜。投票可以随意投，规则上与身份无关。各轮出队人数：5人局2,3,2,3,3；6人局2,3,4,3,4；7人局2,3,3,4,4；8至10人局3,4,4,5,5。\\n【执行任务】每名队员出一张任务牌：好人必须出成功；坏人可以出成功，也可以出失败。牌洗混后翻开，所有人知道失败票的张数，但不知道是谁出的。出现1张失败票任务就失败；例外：7人及以上的第4个任务需要至少2张失败票才失败。\\n【结束】三个任务失败则坏人获胜。三个任务成功后刺客刺杀一人：刺中梅林则坏人获胜，否则好人获胜。\\n【湖中仙女】持有者查验一名玩家，得知其阵营（好人或坏人），结果只有持有者本人知道。\'\'\'\\n\\nTASK = \'\'\'判断给定\\"说法\\"相对于规则和记录（以及你的视角信息）的地位，只能回答一个词：\\n成立：在所有符合规则与记录的身份分配中，说法都为真；\\n不成立：在所有这样的身份分配中，说法都为假；\\n不确定：有的身份分配下为真，有的为假；\\n矛盾：记录或你的视角信息本身违反规则，不存在任何符合的身份分配。\'\'\'\\n\\nANSWER = \'只输出 成立、不成立、不确定、矛盾 中的一个词，不要解释。\'\\n\\nEXTRACT_SYS = \'\'\'把一段《阿瓦隆》局面描述逐字转换成JSON，不做任何推理。格式：\\n{\\"n\\": 人数,\\n \\"config\\": {角色英文名: 数量, ...}（只列出现的角色）,\\n \\"quests\\": [{\\"q\\": 任务编号, \\"team\\": [座位号], \\"fails\\": 失败票张数, \\"result\\": \\"success\\"或\\"fail\\"}]（只列已执行的任务；提案和投票不需要）,\\n \\"view\\": null 或 {\\"player\\": 座位号, \\"role\\": 角色英文名, \\"seen\\": [开局看到的座位号], \\"lady\\": [[被查验的座位号, 是坏人则true否则false]]},\\n \\"claim\\": 说法的JSON}\\n说法的写法：X号一定是坏人 → {\\"t\\":\\"evil\\",\\"p\\":X}；X号一定是好人 → {\\"t\\":\\"good\\",\\"p\\":X}；X号一定是梅林 → {\\"t\\":\\"role\\",\\"p\\":X,\\"r\\":\\"merlin\\"}；X号一定不是梅林 → {\\"t\\":\\"not\\",\\"a\\":{\\"t\\":\\"role\\",\\"p\\":X,\\"r\\":\\"merlin\\"}}；某些人中至少有k名坏人 → {\\"t\\":\\"atleast\\",\\"ps\\":[座位号],\\"k\\":k}；某些人中恰好有k名坏人 → {\\"t\\":\\"exactly\\",\\"ps\\":[座位号],\\"k\\":k}；X号和Y号一定属于同一阵营 → {\\"t\\":\\"same\\",\\"ps\\":[X,Y]}；本局的坏人恰好是某些人 → {\\"t\\":\\"evil_set\\",\\"ps\\":[座位号]}。\\n角色英文名：梅林 merlin，派西维尔 percival，亚瑟的忠臣 servant，莫甘娜 morgana，莫德雷德 mordred，奥伯伦 oberon，莫德雷德的爪牙 minion。座位号使用文字中的号码（从1开始）。旁观者视角时 view 为 null。只输出一个JSON对象，不要解释，不要Markdown。\'\'\'\\n\\nEXTRACT_EXAMPLE_IN = \'\'\'6人局，角色为：梅林、派西维尔、亚瑟的忠臣×2、莫甘娜、莫德雷德的爪牙（共2名坏人）。\\n第1轮第1次提案：队长3号提名1号、6号，赞成者为1号、3号、6号，被否决。\\n第1轮第2次提案：队长4号提名2号、4号，赞成者为1号、2号、4号、5号，通过。\\n第1个任务由2号、4号执行，翻开1张失败票，任务失败。\\n你是5号，身份是派西维尔。开局时你看到1号、3号（其中一人是梅林，另一人是莫甘娜，你分不清）。\\n说法：2号和4号中至少有1名坏人。\'\'\'\\nEXTRACT_EXAMPLE_OUT = (\'{\\"n\\": 6, \\"config\\": {\\"merlin\\": 1, \\"percival\\": 1, \\"servant\\": 2, \\"morgana\\": 1, \\"minion\\": 1}, \'\\n                       \'\\"quests\\": [{\\"q\\": 1, \\"team\\": [2, 4], \\"fails\\": 1, \\"result\\": \\"fail\\"}], \'\\n                       \'\\"view\\": {\\"player\\": 5, \\"role\\": \\"percival\\", \\"seen\\": [1, 3], \\"lady\\": []}, \'\\n                       \'\\"claim\\": {\\"t\\": \\"atleast\\", \\"ps\\": [2, 4], \\"k\\": 1}}\')\\n\\nREVEAL_Q = (\'现在可以用湖中仙女查验若干玩家的阵营（每查验一人，就得知他是好人还是坏人）。\'\\n            \'请一次性给出最少需要查验的玩家，使得无论查验结果如何，都一定能判断上述说法成立还是不成立。\'\\n            \'不能查验你自己。只输出座位号的JSON数组，例如 [3, 5]，不要解释。\')\\n\\n\\ndef digest(x):\\n    return hashlib.sha256(json.dumps(x, ensure_ascii=False, sort_keys=True).encode()).hexdigest()\\n\\n\\ndef user_text(item):\\n    return item[\'text\'] + \'\\\\n\\\\n说法：\' + item[\'claim_text\']\\n\\n\\ndef job(item, task, messages, budget, thinking=False):\\n    return {\'id\': f\'{task}/{item[\\"id\\"]}\', \'item\': item[\'id\'], \'task\': task, \'messages\': messages,\\n            \'max_new_tokens\': budget, \'thinking\': thinking}\\n\\n\\ndef initial_jobs(items):\\n    jobs = []\\n    sys_msg = RULE_CARD + \'\\\\n\\\\n\' + TASK\\n    for it in items:\\n        base = [{\'role\': \'system\', \'content\': sys_msg}, {\'role\': \'user\', \'content\': user_text(it) + \'\\\\n\' + ANSWER}]\\n        jobs.append(job(it, \'direct\', base, 16))\\n        jobs.append(job(it, \'extract\', [{\'role\': \'system\', \'content\': EXTRACT_SYS},\\n                                        {\'role\': \'user\', \'content\': EXTRACT_EXAMPLE_IN},\\n                                        {\'role\': \'assistant\', \'content\': EXTRACT_EXAMPLE_OUT},\\n                                        {\'role\': \'user\', \'content\': user_text(it)}], 768))\\n        if it[\'gold\'] == \'undetermined\' and (it.get(\'gold_reveal\') or {}).get(\'status\') == \'ok\':\\n            jobs.append(job(it, \'reveal\', [{\'role\': \'system\', \'content\': RULE_CARD},\\n                                           {\'role\': \'user\', \'content\': user_text(it) + \'\\\\n\' + REVEAL_Q}], 64))\\n    return jobs\\n\\n\\ndef think_jobs(items, budget=3072):\\n    sys_msg = RULE_CARD + \'\\\\n\\\\n\' + TASK\\n    return [job(it, \'think\', [{\'role\': \'system\', \'content\': sys_msg},\\n                              {\'role\': \'user\', \'content\': user_text(it) + \'\\\\n\' + ANSWER}], budget, thinking=True)\\n            for it in items]\\n\\n\\n# ---------------------------------------------------------------------------\\n# parsing\\n# ---------------------------------------------------------------------------\\ndef strip_think(raw):\\n    return raw.split(\'</think>\')[-1] if \'</think>\' in raw else raw\\n\\n\\ndef parse_label(raw):\\n    t = strip_think(raw).strip().strip(\'。.`*\\"“”\\\\\' \\\\n\')\\n    if t in FROM_WORD:\\n        return FROM_WORD[t], \'strict\'\\n    found = set()\\n    for w in (\'不成立\', \'不确定\', \'矛盾\'):\\n        if w in t:\\n            found.add(FROM_WORD[w])\\n    if \'成立\' in t.replace(\'不成立\', \'\'):\\n        found.add(\'entailed\')\\n    if len(found) == 1:\\n        return found.pop(), \'lenient\'\\n    return \'parse_failure\', None\\n\\n\\ndef _json_block(raw):\\n    t = strip_think(raw).strip()\\n    t = re.sub(r\'^```(?:json)?|```$\', \'\', t.strip(), flags=re.M).strip()\\n    start = t.find(\'{\') if \'{\' in t else t.find(\'[\')\\n    if start < 0:\\n        raise ValueError(\'no json\')\\n    dec = json.JSONDecoder()\\n    obj, _ = dec.raw_decode(t[start:])\\n    return obj\\n\\n\\ndef _claim_to0(c):\\n    c = dict(c)\\n    if c.get(\'t\') in (\'evil\', \'good\', \'role\'):\\n        c[\'p\'] = int(c[\'p\']) - 1\\n    elif c.get(\'t\') in (\'atleast\', \'exactly\', \'same\', \'evil_set\'):\\n        c[\'ps\'] = sorted(int(x) - 1 for x in c[\'ps\'])\\n        if \'k\' in c:\\n            c[\'k\'] = int(c[\'k\'])\\n    elif c.get(\'t\') == \'not\':\\n        c[\'a\'] = _claim_to0(c[\'a\'])\\n    else:\\n        raise ValueError(\'claim type\')\\n    return c\\n\\n\\ndef parse_extraction(raw):\\n    \\"\\"\\"Model JSON (1-based seats) -> (inst, claim) with 0-based seats.\\"\\"\\"\\n    x = _json_block(raw)\\n    n = int(x[\'n\'])\\n    cfg = {str(k): int(v) for k, v in x[\'config\'].items() if int(v) > 0}\\n    if any(k not in ROLES for k in cfg):\\n        raise ValueError(\'role name\')\\n    quests = []\\n    for q in x.get(\'quests\') or []:\\n        res = str(q[\'result\']).lower()\\n        if res not in (\'success\', \'fail\'):\\n            raise ValueError(\'result\')\\n        quests.append({\'q\': int(q[\'q\']), \'team\': sorted(int(i) - 1 for i in q[\'team\']),\\n                       \'fails\': None if q.get(\'fails\') is None else int(q[\'fails\']), \'result\': res, \'proposals\': []})\\n    v = x.get(\'view\')\\n    view = None\\n    if v:\\n        view = {\'player\': int(v[\'player\']) - 1, \'role\': str(v[\'role\']),\\n                \'seen\': sorted(int(i) - 1 for i in v.get(\'seen\') or []),\\n                \'lady\': [[int(t) - 1, bool(e)] for t, e in v.get(\'lady\') or []]}\\n        if view[\'role\'] not in ROLES:\\n            raise ValueError(\'view role\')\\n    inst = {\'rec\': {\'n\': n, \'config\': cfg, \'quests\': quests, \'end\': None}, \'view\': view}\\n    return inst, _claim_to0(x[\'claim\'])\\n\\n\\ndef parse_reveal(raw, cands):\\n    x = _json_block(raw)\\n    if not isinstance(x, list) or any(type(i) is not int for i in x):\\n        raise ValueError(\'not an int array\')\\n    s = sorted(i - 1 for i in x)\\n    if len(set(s)) != len(s):\\n        raise ValueError(\'duplicate\')\\n    if any(i not in cands for i in s):\\n        raise ValueError(\'not a candidate\')\\n    return s\\n\\n\\ndef gold_shape(item):\\n    \\"\\"\\"Extraction target: the item\'s formalisation without proposals.\\"\\"\\"\\n    rec = item[\'inst\'][\'rec\']\\n    quests = [{\'q\': q[\'q\'], \'team\': sorted(q[\'team\']), \'fails\': q[\'fails\'], \'result\': q[\'result\']}\\n              for q in rec[\'quests\'] if q.get(\'result\') is not None]\\n    return {\'n\': rec[\'n\'], \'config\': {k: v for k, v in rec[\'config\'].items() if v}, \'quests\': quests,\\n            \'view\': item[\'inst\'][\'view\'], \'claim\': item[\'claim\']}\\n\\n\\ndef shape_of(inst, claim):\\n    rec = inst[\'rec\']\\n    return {\'n\': rec[\'n\'], \'config\': rec[\'config\'],\\n            \'quests\': [{\'q\': q[\'q\'], \'team\': q[\'team\'], \'fails\': q[\'fails\'], \'result\': q[\'result\']} for q in rec[\'quests\']],\\n            \'view\': inst[\'view\'], \'claim\': claim}\\n\\n\\ndef _norm_view(v):\\n    if v is None:\\n        return None\\n    return {\'player\': v[\'player\'], \'role\': v[\'role\'], \'seen\': sorted(v.get(\'seen\') or []),\\n            \'lady\': sorted([list(map(int, x[:1])) + [bool(x[1])] for x in v.get(\'lady\') or []])}\\n\\n\\ndef _norm_claim(c):\\n    c = dict(c)\\n    if \'ps\' in c:\\n        c[\'ps\'] = sorted(c[\'ps\'])\\n    if c.get(\'t\') == \'not\':\\n        c[\'a\'] = _norm_claim(c[\'a\'])\\n    return c\\n\\n\\n# ---------------------------------------------------------------------------\\n# solver feedback\\n# ---------------------------------------------------------------------------\\ndef render_world(roles):\\n    return \'，\'.join(f\'{player_label(i)}{ZH_ROLE[r]}\' for i, r in enumerate(roles))\\n\\n\\ndef render_core(tags, item):\\n    rec = item[\'inst\'][\'rec\']\\n    cfg, n = rec[\'config\'], rec[\'n\']\\n    out = []\\n    for tag in tags:\\n        rid, _, where = tag.partition(\'@\')\\n        if rid.startswith(\'S-DEAL\'):\\n            r = rid[rid.index(\'[\') + 1:-1]\\n            out.append(f\'本局{ZH_ROLE[r]}共{cfg.get(r, 0)}名\')\\n        elif rid == \'Q-COUNT\':\\n            q = int(where[1:])\\n            f = next(x[\'fails\'] for x in rec[\'quests\'] if x[\'q\'] == q)\\n            out.append(f\'第{q}个任务翻开了{f}张失败票\')\\n        elif rid == \'Q-GOOD\':\\n            q, i = where[1:].split(\':\')\\n            out.append(f\'好人必须出成功票（第{q}个任务中的{int(i) + 1}号）\')\\n        elif rid == \'Q-RESULT\':\\n            q = int(where[1:])\\n            out.append(f\'第{q}个任务需要至少{fails_required(n, q)}张失败票才算失败\')\\n        elif rid == \'Q-SIZE\':\\n            q = int(where[1:])\\n            out.append(f\'第{q}个任务应由{TEAM_SIZE[n][q - 1]}人执行\')\\n        elif rid == \'V-SELF\':\\n            out.append(\'你自己的身份\')\\n        elif rid in (\'K-MERLIN\', \'K-PERCIVAL\', \'K-EVIL\', \'K-NONE\'):\\n            who = {\'K-MERLIN\': \'梅林\', \'K-PERCIVAL\': \'派西维尔\', \'K-EVIL\': \'坏人\', \'K-NONE\': \'该身份\'}[rid]\\n            out.append(f\'{who}的开局视野（关于{int(where) + 1}号）\')\\n        elif rid == \'K-SELF\':\\n            out.append(\'开局视野不包括自己\')\\n        elif rid == \'K-LADY\':\\n            out.append(f\'湖中仙女对{int(where) + 1}号的查验结果\')\\n        elif rid == \'S-COUNT\':\\n            out.append(\'角色配置中的坏人数量必须符合人数表\')\\n        else:\\n            out.append(f\'规则{rid}\' + (f\'（{where}）\' if where else \'\'))\\n    return \'；\'.join(out) + \'。\'\\n\\n\\ndef feedback(item, answer_label, smt, mode):\\n    head = f\'约束求解器检查了你的回答“{WORD.get(answer_label, answer_label)}”，发现它不被规则与记录支持。\'\\n    if mode == \'flag\':\\n        return head + \'请重新判断。\' + ANSWER\\n    res = smt.classify(item[\'inst\'], item[\'claim\'], explain=True)\\n    g = res[\'label\']\\n    if g == \'undetermined\':\\n        ev = (\'下面两种身份分配都符合规则与记录：\\\\n情形一（说法为真）：\' + render_world(res[\'positive\'])\\n              + \'\\\\n情形二（说法为假）：\' + render_world(res[\'negative\']))\\n    elif g == \'inconsistent\':\\n        ev = \'记录本身违反规则，冲突来自：\' + render_core(res[\'core\'], item)\\n    else:\\n        w = res[\'positive\'] or res[\'negative\']\\n        ev = (\'存在符合规则与记录的身份分配，例如：\' + render_world(w) + \'\\\\n\') if answer_label == \'inconsistent\' else \'\'\\n        ev += (\'不存在使说法为假的身份分配\' if g == \'entailed\' else \'不存在使说法为真的身份分配\')\\n        ev += \'，推理所需的规则与记录：\' + render_core(res[\'core\'], item)\\n    return head + \'\\\\n\' + ev + \'\\\\n请重新判断。\' + ANSWER\\n\\n\\ndef revise_jobs(items, direct_records, smt, mode):\\n    by = {r[\'item\']: r for r in direct_records}\\n    jobs = []\\n    sys_msg = RULE_CARD + \'\\\\n\\\\n\' + TASK\\n    for it in items:\\n        r = by[it[\'id\']]\\n        lab, _ = parse_label(r[\'raw\'])\\n        if lab == it[\'gold\']:\\n            continue\\n        msgs = [{\'role\': \'system\', \'content\': sys_msg},\\n                {\'role\': \'user\', \'content\': user_text(it) + \'\\\\n\' + ANSWER},\\n                {\'role\': \'assistant\', \'content\': r[\'raw\'].strip() or \'（无输出）\'},\\n                {\'role\': \'user\', \'content\': feedback(it, lab, smt, mode)}]\\n        jobs.append(job(it, \'revise_\' + mode, msgs, 16))\\n    return jobs\\n\\n\\n# ---------------------------------------------------------------------------\\n# model runner\\n# ---------------------------------------------------------------------------\\nclass Runner:\\n    def __init__(self, out_dir, batch_size=None):\\n        import torch\\n        import transformers\\n        import platform\\n        from transformers import AutoTokenizer, AutoModelForCausalLM\\n        assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), \'Need a BF16-capable CUDA GPU runtime\'\\n        self.torch = torch\\n        mem = torch.cuda.get_device_properties(0).total_memory / 2 ** 30\\n        self.batch = batch_size or (16 if mem > 60 else 8)\\n        self.tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, padding_side=\'left\')\\n        if self.tok.pad_token_id is None:\\n            self.tok.pad_token = self.tok.eos_token\\n        self.model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, torch_dtype=torch.bfloat16,\\n                                                          attn_implementation=\'sdpa\').to(\'cuda\').eval()\\n        self.out = Path(out_dir)\\n        self.out.mkdir(parents=True, exist_ok=True)\\n        self.meta = {\'model\': MODEL, \'revision\': REVISION, \'gpu\': torch.cuda.get_device_name(0),\\n                     \'gpu_mem_gib\': round(mem, 1), \'python\': platform.python_version(), \'torch\': torch.__version__,\\n                     \'transformers\': transformers.__version__, \'dtype\': \'bfloat16\', \'batch_size\': self.batch,\\n                     \'greedy_when_thinking_off\': True,\\n                     \'thinking_sampling\': {\'temperature\': 0.6, \'top_p\': 0.95, \'top_k\': 20},\\n                     \'seed\': 42, \'code_sha256\': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}\\n        (self.out / \'runner_metadata.json\').write_text(json.dumps(self.meta, indent=2), encoding=\'utf-8\')\\n\\n    def run(self, jobs, name, batch_size=None):\\n        \\"\\"\\"A job may carry \'prefix\': text appended after the chat template\'s generation prompt\\n        (used by followup.py to continue or budget-force a thinking trace).\\"\\"\\"\\n        meta = {\'jobs_sha256\': digest(jobs), \'n\': len(jobs)}\\n        mp, rp = self.out / f\'{name}_metadata.json\', self.out / f\'{name}_records.jsonl\'\\n        if mp.exists():\\n            assert json.loads(mp.read_text()) == meta, \'resume mismatch: use a new name\'\\n        else:\\n            mp.write_text(json.dumps(meta, indent=2), encoding=\'utf-8\')\\n        (self.out / f\'{name}_jobs.json\').write_text(json.dumps(jobs, ensure_ascii=False), encoding=\'utf-8\')\\n        records = [json.loads(x) for x in rp.read_text(encoding=\'utf-8\').splitlines() if x.strip()] if rp.exists() else []\\n        done = {r[\'id\'] for r in records}\\n        groups = sorted({(j[\'max_new_tokens\'], j[\'thinking\']) for j in jobs})\\n        for budget, thinking in groups:\\n            todo = [j for j in jobs if j[\'id\'] not in done and j[\'max_new_tokens\'] == budget and j[\'thinking\'] == thinking]\\n            bs = batch_size or (self.batch if budget <= 1024 else max(4, self.batch))\\n            for b0 in range(0, len(todo), bs):\\n                batch = todo[b0:b0 + bs]\\n                texts = [self.tok.apply_chat_template(j[\'messages\'], tokenize=False, add_generation_prompt=True,\\n                                                      enable_thinking=thinking) + j.get(\'prefix\', \'\') for j in batch]\\n                enc = self.tok(texts, return_tensors=\'pt\', padding=True).to(\'cuda\')\\n                kw = dict(max_new_tokens=budget, pad_token_id=self.tok.pad_token_id)\\n                if thinking:\\n                    self.torch.manual_seed(42 + b0)\\n                    kw.update(do_sample=True, temperature=0.6, top_p=0.95, top_k=20)\\n                else:\\n                    kw.update(do_sample=False)\\n                self.torch.cuda.synchronize()\\n                t0 = time.perf_counter()\\n                with self.torch.inference_mode():\\n                    out = self.model.generate(**enc, **kw)\\n                self.torch.cuda.synchronize()\\n                sec = time.perf_counter() - t0\\n                gen = out[:, enc.input_ids.shape[1]:]\\n                dec = self.tok.batch_decode(gen, skip_special_tokens=True)\\n                with rp.open(\'a\', encoding=\'utf-8\') as f:\\n                    for j, raw, ids in zip(batch, dec, gen):\\n                        ids = ids.tolist()\\n                        eos = self.tok.eos_token_id\\n                        row = {\'id\': j[\'id\'], \'item\': j[\'item\'], \'task\': j[\'task\'], \'raw\': raw,\\n                               \'output_tokens\': ids.index(eos) + 1 if eos in ids else len(ids),\\n                               \'hit_limit\': eos not in ids, \'batch_seconds\': sec, \'batch_size\': len(batch),\\n                               \'input_tokens\': int(enc.input_ids.shape[1])}\\n                        f.write(json.dumps(row, ensure_ascii=False) + \'\\\\n\')\\n                        records.append(row)\\n                print(f\'{name} budget={budget} think={thinking}: {b0 + len(batch)}/{len(todo)} {sec:.1f}s\', flush=True)\\n        assert len({r[\'id\'] for r in records}) == len(jobs)\\n        return records\\n\\n\\n# ---------------------------------------------------------------------------\\n# scoring\\n# ---------------------------------------------------------------------------\\ndef cls_metrics(rows):\\n    n = len(rows)\\n    conf = {g: Counter() for g in LABELS}\\n    for r in rows:\\n        conf[r[\'gold\']][r[\'pred\']] += 1\\n    f1 = []\\n    for k in LABELS:\\n        tp = conf[k][k]\\n        fp = sum(conf[g][k] for g in LABELS if g != k)\\n        fn = sum(v for p, v in conf[k].items() if p != k)\\n        f1.append(2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0)\\n    und = [r for r in rows if r[\'gold\'] == \'undetermined\']\\n    dec = [r for r in rows if r[\'gold\'] in (\'entailed\', \'contradicted\')]\\n    inc = [r for r in rows if r[\'gold\'] == \'inconsistent\']\\n    return {\'n\': n, \'accuracy\': sum(r[\'gold\'] == r[\'pred\'] for r in rows) / n if n else None,\\n            \'macro_f1\': sum(f1) / 4, \'f1\': dict(zip(LABELS, f1)),\\n            \'confusion\': {g: dict(c) for g, c in conf.items()},\\n            \'premature_certainty\': [sum(r[\'pred\'] in (\'entailed\', \'contradicted\') for r in und), len(und)],\\n            \'false_uncertainty\': [sum(r[\'pred\'] == \'undetermined\' for r in dec), len(dec)],\\n            \'missed_inconsistency\': [sum(r[\'pred\'] != \'inconsistent\' for r in inc), len(inc)],\\n            \'false_inconsistency\': [sum(r[\'pred\'] == \'inconsistent\' for r in rows if r[\'gold\'] != \'inconsistent\'),\\n                                    sum(r[\'gold\'] != \'inconsistent\' for r in rows)],\\n            \'parse_failures\': sum(r[\'pred\'] == \'parse_failure\' for r in rows)}\\n\\n\\ndef bootstrap_diff(a, b, seed=271, reps=2000):\\n    \\"\\"\\"Paired bootstrap over items of mean(b) - mean(a); a, b are 0/1 lists.\\"\\"\\"\\n    rng = random.Random(seed)\\n    n = len(a)\\n    diffs = []\\n    for _ in range(reps):\\n        idx = [rng.randrange(n) for _ in range(n)]\\n        diffs.append(sum(b[i] - a[i] for i in idx) / n)\\n    diffs.sort()\\n    return {\'diff\': sum(b[i] - a[i] for i in range(n)) / n, \'ci95\': [diffs[int(0.025 * reps)], diffs[int(0.975 * reps) - 1]]}\\n\\n\\ndef strata(items):\\n    def pkind(p):\\n        return \'observer\' if p == \'observer\' else (\'evil_view\' if p in EVIL_ROLES else \'good_view\')\\n    return {it[\'id\']: {\'perspective\': pkind(it[\'perspective\']), \'claim\': it[\'claim\'][\'t\'],\\n                       \'n\': it[\'inst\'][\'rec\'][\'n\'], \'corruption\': it.get(\'corruption\') or \'-\'} for it in items}\\n\\n\\ndef score(items, recs, smt):\\n    by = defaultdict(dict)\\n    for task, rr in recs.items():\\n        for r in rr:\\n            by[task][r[\'item\']] = r\\n    st = strata(items)\\n    out, correct = {}, {}\\n    for task in (\'direct\', \'think\'):\\n        if task not in by:\\n            continue\\n        rows = []\\n        for it in items:\\n            lab, mode = parse_label(by[task][it[\'id\']][\'raw\'])\\n            rows.append({\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': lab, \'mode\': mode})\\n        m = cls_metrics(rows)\\n        m[\'lenient_parses\'] = sum(r[\'mode\'] == \'lenient\' for r in rows)\\n        m[\'hit_token_limit\'] = sum(by[task][it[\'id\']].get(\'hit_limit\', False) for it in items)\\n        m[\'mean_output_tokens\'] = sum(by[task][it[\'id\']][\'output_tokens\'] for it in items) / len(items)\\n        m[\'by\'] = {k: {v: cls_metrics([r for r in rows if st[r[\'id\']][k] == v])[\'accuracy\']\\n                       for v in sorted({s[k] for s in st.values()}, key=str)} for k in (\'perspective\', \'claim\', \'n\')}\\n        out[task] = m\\n        correct[task] = [int(r[\'gold\'] == r[\'pred\']) for r in rows]\\n    # extraction + solver\\n    if \'extract\' in by:\\n        rows, ex = [], Counter()\\n        for it in items:\\n            raw = by[\'extract\'][it[\'id\']][\'raw\']\\n            try:\\n                inst, claim = parse_extraction(raw)\\n                ex[\'valid_json\'] += 1\\n                gs, ps = gold_shape(it), shape_of(inst, claim)\\n                ex[\'n_ok\'] += gs[\'n\'] == ps[\'n\']\\n                ex[\'config_ok\'] += gs[\'config\'] == ps[\'config\']\\n                ex[\'quests_ok\'] += gs[\'quests\'] == ps[\'quests\']\\n                ex[\'view_ok\'] += _norm_view(gs[\'view\']) == _norm_view(ps[\'view\'])\\n                ex[\'claim_ok\'] += _norm_claim(gs[\'claim\']) == _norm_claim(ps[\'claim\'])\\n                exact = (gs[\'n\'] == ps[\'n\'] and gs[\'config\'] == ps[\'config\'] and gs[\'quests\'] == ps[\'quests\']\\n                         and _norm_view(gs[\'view\']) == _norm_view(ps[\'view\']) and _norm_claim(gs[\'claim\']) == _norm_claim(ps[\'claim\']))\\n                ex[\'exact\'] += exact\\n                try:\\n                    lab = smt.classify(inst, claim)[\'label\']\\n                except Exception:\\n                    lab = \'parse_failure\'\\n                    ex[\'solver_error\'] += 1\\n            except Exception:\\n                lab = \'parse_failure\'\\n            rows.append({\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': lab})\\n        m = cls_metrics(rows)\\n        m[\'extraction\'] = dict(ex)\\n        m[\'by\'] = {k: {v: cls_metrics([r for r in rows if st[r[\'id\']][k] == v])[\'accuracy\']\\n                       for v in sorted({s[k] for s in st.values()}, key=str)} for k in (\'perspective\', \'claim\', \'n\')}\\n        out[\'extract_solve\'] = m\\n        correct[\'extract_solve\'] = [int(r[\'gold\'] == r[\'pred\']) for r in rows]\\n    # revision with solver feedback (only wrong direct answers were re-asked)\\n    for mode in (\'flag\', \'evidence\'):\\n        t = \'revise_\' + mode\\n        if t not in by:\\n            continue\\n        rows, fixed = [], 0\\n        for it in items:\\n            lab, _ = parse_label(by[\'direct\'][it[\'id\']][\'raw\'])\\n            if lab != it[\'gold\']:\\n                lab2, _ = parse_label(by[t][it[\'id\']][\'raw\'])\\n                fixed += lab2 == it[\'gold\']\\n                lab = lab2\\n            rows.append({\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': lab})\\n        m = cls_metrics(rows)\\n        m[\'reasked\'] = len(by[t])\\n        m[\'fixed\'] = fixed\\n        out[t] = m\\n        correct[t] = [int(r[\'gold\'] == r[\'pred\']) for r in rows]\\n    # reveal selection\\n    if \'reveal\' in by:\\n        from search import candidates\\n        rv = Counter()\\n        excess = []\\n        for it in items:\\n            if it[\'id\'] not in by[\'reveal\']:\\n                continue\\n            rv[\'n\'] += 1\\n            cands = candidates(it[\'inst\'])\\n            try:\\n                S = parse_reveal(by[\'reveal\'][it[\'id\']][\'raw\'], cands)\\n            except Exception:\\n                rv[\'invalid\'] += 1\\n                continue\\n            suff = smt.find_pair(it[\'inst\'], it[\'claim\'], S) is None\\n            rv[\'sufficient\'] += suff\\n            gsize = it[\'gold_reveal\'][\'size\']\\n            if suff:\\n                rv[\'optimal\'] += len(S) == gsize\\n                excess.append(len(S) - gsize)\\n            rv[\'all_candidates_chosen\'] += len(S) == len(cands)\\n        out[\'reveal\'] = {**rv, \'mean_excess_when_sufficient\': sum(excess) / len(excess) if excess else None,\\n                         \'gold_total\': sum(it[\'gold_reveal\'][\'size\'] for it in items if it[\'id\'] in by[\'reveal\'])}\\n    # end-to-end verification of direct answers with the model\'s own formalisation\\n    if \'direct\' in by and \'extract\' in by:\\n        flagged_wrong = flagged_right = wrong = right = 0\\n        for it in items:\\n            lab, _ = parse_label(by[\'direct\'][it[\'id\']][\'raw\'])\\n            try:\\n                inst, claim = parse_extraction(by[\'extract\'][it[\'id\']][\'raw\'])\\n                verdict = smt.classify(inst, claim)[\'label\']\\n            except Exception:\\n                verdict = None\\n            flag = verdict is not None and verdict != lab\\n            if lab == it[\'gold\']:\\n                right += 1\\n                flagged_right += flag\\n            else:\\n                wrong += 1\\n                flagged_wrong += flag\\n        out[\'verification_pipeline\'] = {\'direct_wrong\': wrong, \'wrong_flagged\': flagged_wrong,\\n                                         \'direct_right\': right, \'right_flagged\': flagged_right}\\n    comps = {}\\n    for t in (\'think\', \'extract_solve\', \'revise_flag\', \'revise_evidence\'):\\n        if t in correct and \'direct\' in correct:\\n            comps[f\'{t}_minus_direct\'] = bootstrap_diff(correct[\'direct\'], correct[t])\\n    out[\'paired_bootstrap\'] = comps\\n    return out\\n", "avalon/followup.py": "\\"\\"\\"Follow-up experiments on the frozen Avalon task set (protocol: CODEX_FOLLOWUP.md).\\n\\nRe-uses the first run\'s records (direct / extract / think@3072) unchanged and adds:\\n\\n  think budget    Every thinking trace cut off at 3072 new tokens is *continued* from its own\\n                  text up to 8192, then (if still unfinished) 16384 new tokens in total.\\n                  Sampling just carries on from the same prefix, so a continued trace is a\\n                  sample of what one run with the larger budget would have produced; traces\\n                  that already finished are unchanged.  At every budget, traces that are still\\n                  unfinished are additionally budget-forced (close </think>, ask for the verdict).\\n  verify_*        The realistic pipeline (no gold label is read anywhere): re-ask only when the\\n                  direct answer disagrees with Z3 run on the model\'s OWN extraction, with the\\n                  feedback computed from that extraction (\'flag\' or \'evidence\').\\n  extract_repair  Logic-LM style: when Z3 finds the model\'s extraction unsatisfiable (or it\\n                  cannot be used), show the conflicting clauses once and ask for a re-check.\\n  verify_evidence_repaired   verify_evidence on top of the repaired extraction.\\n\\nJob builders only ever receive blind(items): gold labels, true roles and gold reveal sets removed.\\n\\n  python avalon/followup.py --selftest                  # local, no GPU: prompts, job counts, scorer\\n  python avalon/followup.py --score --base DIR --new DIR --out FILE\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport random\\nfrom collections import Counter\\nfrom pathlib import Path\\n\\nfrom llm_run import (RULE_CARD, TASK, ANSWER, EXTRACT_SYS, EXTRACT_EXAMPLE_IN, EXTRACT_EXAMPLE_OUT, LABELS,\\n                     user_text, job, think_jobs, initial_jobs, parse_label, parse_extraction, feedback, render_core,\\n                     cls_metrics, bootstrap_diff, gold_shape, shape_of, _norm_view, _norm_claim, digest, score)\\n\\nBUDGETS = (3072, 8192, 16384)\\nFORCE = \'\\\\n\\\\n考虑到思考时间有限，我现在必须根据已有的思考直接给出最终答案。\\\\n</think>\\\\n\\\\n\'\\nFORCE_TOKENS = 16\\nHIDDEN = (\'gold\', \'true_roles\', \'gold_reveal\')\\nSYS = RULE_CARD + \'\\\\n\\\\n\' + TASK\\nKV_GIB_PER_TOKEN = 2 * 36 * 8 * 128 * 2 / 2 ** 30   # Qwen3-8B: 36 layers, 8 KV heads, dim 128, bf16\\n\\n\\ndef blind(items):\\n    return [{k: v for k, v in it.items() if k not in HIDDEN} for it in items]\\n\\n\\ndef by_item(records, task=None):\\n    return {r[\'item\']: r for r in records if task is None or r[\'task\'] == task}\\n\\n\\ndef read_jsonl(p):\\n    return [json.loads(x) for x in Path(p).read_text(encoding=\'utf-8\').splitlines() if x.strip()]\\n\\n\\ndef batch_size_for(mem_gib, max_context):\\n    \\"\\"\\"Largest safe batch for long thinking continuations (KV cache + 18 GiB for weights/overhead).\\"\\"\\"\\n    per_seq = max_context * KV_GIB_PER_TOKEN\\n    return max(1, min(32, int(0.6 * (mem_gib - 18) / per_seq)))\\n\\n\\n# ---------------------------------------------------------------------------\\n# frozen prompts\\n# ---------------------------------------------------------------------------\\ndef extract_messages(it):\\n    \\"\\"\\"Identical to the \'extract\' job of llm_run.initial_jobs (checked in check_frozen).\\"\\"\\"\\n    return [{\'role\': \'system\', \'content\': EXTRACT_SYS},\\n            {\'role\': \'user\', \'content\': EXTRACT_EXAMPLE_IN},\\n            {\'role\': \'assistant\', \'content\': EXTRACT_EXAMPLE_OUT},\\n            {\'role\': \'user\', \'content\': user_text(it)}]\\n\\n\\ndef check_frozen(items, initial_meta, think_meta):\\n    \\"\\"\\"The first run\'s prompts are reproduced byte for byte (same job digests).\\"\\"\\"\\n    ij = initial_jobs(items)\\n    assert digest(ij) == initial_meta[\'jobs_sha256\'], \'initial prompts differ from the first run\'\\n    assert digest(think_jobs(items, budget=3072)) == think_meta[\'jobs_sha256\'], \'thinking prompts differ\'\\n    em = {j[\'item\']: j[\'messages\'] for j in ij if j[\'task\'] == \'extract\'}\\n    assert all(extract_messages(it) == em[it[\'id\']] for it in blind(items))\\n    return True\\n\\n\\n# ---------------------------------------------------------------------------\\n# the model\'s own formalisation\\n# ---------------------------------------------------------------------------\\ndef formalise(raw, smt):\\n    \\"\\"\\"Extraction text -> {\'inst\', \'claim\', \'label\'}; \'label\' is missing when Z3 cannot use it.\\"\\"\\"\\n    try:\\n        inst, claim = parse_extraction(raw)\\n    except Exception as e:\\n        return {\'error\': \'parse:\' + type(e).__name__}\\n    try:\\n        return {\'inst\': inst, \'claim\': claim, \'label\': smt.classify(inst, claim)[\'label\']}\\n    except Exception as e:\\n        return {\'inst\': inst, \'claim\': claim, \'error\': \'solver:\' + type(e).__name__}\\n\\n\\ndef formalise_all(bitems, extract, smt):\\n    return {it[\'id\']: formalise(extract[it[\'id\']][\'raw\'], smt) for it in bitems}\\n\\n\\ndef verify_jobs(bitems, direct, forms, smt, mode, task):\\n    \\"\\"\\"Re-ask only where the direct answer disagrees with Z3 on the model\'s own extraction.\\"\\"\\"\\n    jobs = []\\n    for it in bitems:\\n        lab, _ = parse_label(direct[it[\'id\']][\'raw\'])\\n        f = forms[it[\'id\']]\\n        if \'label\' not in f or f[\'label\'] == lab:\\n            continue\\n        fb = feedback({\'inst\': f[\'inst\'], \'claim\': f[\'claim\']}, lab, smt, mode)\\n        msgs = [{\'role\': \'system\', \'content\': SYS},\\n                {\'role\': \'user\', \'content\': user_text(it) + \'\\\\n\' + ANSWER},\\n                {\'role\': \'assistant\', \'content\': direct[it[\'id\']][\'raw\'].strip() or \'（无输出）\'},\\n                {\'role\': \'user\', \'content\': fb}]\\n        jobs.append(job(it, task, msgs, 16))\\n    return jobs\\n\\n\\nEXTRA_CLAUSE = {\\n    \'S-SIZE\': \'人数必须在5到10之间\',\\n    \'S-TOTAL\': \'角色配置的总人数必须等于玩家人数\',\\n    \'S-UNIQUE\': \'梅林、派西维尔、莫甘娜、莫德雷德、奥伯伦各至多一名\',\\n    \'R-ORDER\': \'任务必须按第1、2、3……个的顺序记录\',\\n    \'G-END\': \'三个任务成功或三个任务失败后游戏结束，之后不再有任务\',\\n}\\nREPAIR_TAIL = (\'请逐项对照原文检查你的转写：人数、角色配置、每个任务的队员、失败票张数和结果、你的视角信息、说法。\'\\n               \'如果转写有误，输出改正后的完整JSON；如果转写与原文一致（也就是原文记录本身违反规则），就原样输出同一个JSON。\'\\n               \'只输出一个JSON对象，不要解释，不要Markdown。\')\\n\\n\\ndef render_core_ext(tags, inst):\\n    item, parts = {\'inst\': inst}, []\\n    for tag in tags:\\n        rid, _, where = tag.partition(\'@\')\\n        if rid in EXTRA_CLAUSE:\\n            parts.append(EXTRA_CLAUSE[rid] + (f\'（{where}）\' if where else \'\'))\\n            continue\\n        try:\\n            parts.append(render_core([tag], item).rstrip(\'。\'))\\n        except Exception:\\n            parts.append(f\'规则{tag}\')\\n    return \'；\'.join(parts) + \'。\'\\n\\n\\ndef repair_feedback(f, smt):\\n    if \'inst\' not in f:\\n        return \'你的输出无法解析为要求格式的JSON。\' + REPAIR_TAIL\\n    if \'label\' not in f:\\n        return \'你的JSON中有约束求解器无法处理的内容（例如超出人数范围的座位号）。\' + REPAIR_TAIL\\n    core = smt.classify(f[\'inst\'], f[\'claim\'], explain=True)[\'core\']\\n    return (\'把你的JSON交给约束求解器后，发现不存在任何符合规则的身份分配。冲突来自：\'\\n            + render_core_ext(core, f[\'inst\']) + \'\\\\n\' + REPAIR_TAIL)\\n\\n\\ndef repair_jobs(bitems, extract, forms, smt):\\n    jobs = []\\n    for it in bitems:\\n        f = forms[it[\'id\']]\\n        if f.get(\'label\') not in (None, \'inconsistent\'):\\n            continue\\n        msgs = extract_messages(it) + [\\n            {\'role\': \'assistant\', \'content\': extract[it[\'id\']][\'raw\'].strip() or \'（无输出）\'},\\n            {\'role\': \'user\', \'content\': repair_feedback(f, smt)}]\\n        jobs.append(job(it, \'extract_repair\', msgs, 768))\\n    return jobs\\n\\n\\ndef repaired_forms(bitems, forms, repair, smt):\\n    \\"\\"\\"A repair that does not parse as JSON keeps the original extraction (fixed in advance).\\"\\"\\"\\n    out = {}\\n    for it in bitems:\\n        r = repair.get(it[\'id\'])\\n        g = formalise(r[\'raw\'], smt) if r else None\\n        out[it[\'id\']] = g if g is not None and \'inst\' in g else forms[it[\'id\']]\\n    return out\\n\\n\\n# ---------------------------------------------------------------------------\\n# thinking budget: continuation and budget forcing\\n# ---------------------------------------------------------------------------\\ndef initial_traces(think_records):\\n    return {r[\'item\']: {\'raw\': r[\'raw\'], \'tokens\': r[\'output_tokens\'], \'hit_limit\': r[\'hit_limit\']}\\n            for r in think_records}\\n\\n\\ndef _think_messages(bitems):\\n    return {j[\'item\']: j[\'messages\'] for j in think_jobs(bitems)}\\n\\n\\ndef continue_jobs(bitems, traces, budget, task):\\n    msgs, jobs = _think_messages(bitems), []\\n    for it in bitems:\\n        t = traces[it[\'id\']]\\n        if not t[\'hit_limit\']:\\n            continue\\n        assert t[\'tokens\'] < budget\\n        j = job(it, task, msgs[it[\'id\']], budget - t[\'tokens\'], thinking=True)\\n        j[\'prefix\'] = t[\'raw\'].rstrip(\'�\')   # drop a dangling partial UTF-8 character\\n        jobs.append(j)\\n    return jobs\\n\\n\\ndef extend(traces, records):\\n    out = dict(traces)\\n    for r in records:\\n        t = traces[r[\'item\']]\\n        out[r[\'item\']] = {\'raw\': t[\'raw\'].rstrip(\'�\') + r[\'raw\'], \'tokens\': t[\'tokens\'] + r[\'output_tokens\'],\\n                          \'hit_limit\': r[\'hit_limit\']}\\n    return out\\n\\n\\ndef force_prefix(raw):\\n    raw = raw.rstrip(\'�\')\\n    return raw if \'</think>\' in raw else raw + FORCE\\n\\n\\ndef force_jobs(bitems, traces, task):\\n    msgs, jobs = _think_messages(bitems), []\\n    for it in bitems:\\n        t = traces[it[\'id\']]\\n        if not t[\'hit_limit\']:\\n            continue\\n        j = job(it, task, msgs[it[\'id\']], FORCE_TOKENS, thinking=True)\\n        j[\'prefix\'] = force_prefix(t[\'raw\'])\\n        jobs.append(j)\\n    return jobs\\n\\n\\ndef parse_think(raw):\\n    \\"\\"\\"Thinking-mode verdict: only text after </think> counts (an unfinished trace is a parse failure).\\"\\"\\"\\n    if \'</think>\' not in raw:\\n        return \'parse_failure\', None\\n    return parse_label(raw)\\n\\n\\ndef think_traces(base_think, new):\\n    \\"\\"\\"traces[B] and forced[B] (item -> full forced text) for every budget.\\"\\"\\"\\n    traces = {BUDGETS[0]: initial_traces(base_think)}\\n    for prev, b in zip(BUDGETS, BUDGETS[1:]):\\n        traces[b] = extend(traces[prev], new.get(f\'think_cont{b}\', []))\\n    forced = {}\\n    for b in BUDGETS:\\n        forced[b] = {r[\'item\']: force_prefix(traces[b][r[\'item\']][\'raw\']) + r[\'raw\']\\n                     for r in new.get(f\'think_force{b}\', [])}\\n    return traces, forced\\n\\n\\n# ---------------------------------------------------------------------------\\n# scoring\\n# ---------------------------------------------------------------------------\\ndef extraction_quality(items, forms):\\n    q = Counter()\\n    for it in items:\\n        f = forms[it[\'id\']]\\n        if \'inst\' not in f:\\n            continue\\n        gs, ps = gold_shape(it), shape_of(f[\'inst\'], f[\'claim\'])\\n        ok = {\'n_ok\': gs[\'n\'] == ps[\'n\'], \'config_ok\': gs[\'config\'] == ps[\'config\'],\\n              \'quests_ok\': gs[\'quests\'] == ps[\'quests\'],\\n              \'view_ok\': _norm_view(gs[\'view\']) == _norm_view(ps[\'view\']),\\n              \'claim_ok\': _norm_claim(gs[\'claim\']) == _norm_claim(ps[\'claim\'])}\\n        q[\'valid_json\'] += 1\\n        for k, v in ok.items():\\n            q[k] += v\\n        q[\'exact\'] += all(ok.values())\\n    return dict(q)\\n\\n\\ndef _verify_condition(items, direct, forms, recs):\\n    rows, s = [], Counter()\\n    for it in items:\\n        d, _ = parse_label(direct[it[\'id\']][\'raw\'])\\n        pred = d\\n        if it[\'id\'] in recs:\\n            v = forms[it[\'id\']][\'label\']\\n            pred, _ = parse_label(recs[it[\'id\']][\'raw\'])\\n            s[\'reasked\'] += 1\\n            s[\'fixed\'] += d != it[\'gold\'] and pred == it[\'gold\']\\n            s[\'broken\'] += d == it[\'gold\'] and pred != it[\'gold\']\\n            s[\'followed_solver\'] += pred == v\\n            s[\'kept_own_answer\'] += pred == d\\n            key = \'solver_right\' if v == it[\'gold\'] else \'solver_wrong\'\\n            s[key] += 1\\n            s[\'followed_when_\' + key] += pred == v\\n        rows.append({\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': pred})\\n    m = cls_metrics(rows)\\n    m.update({k: s[k] for k in (\'reasked\', \'fixed\', \'broken\', \'followed_solver\', \'kept_own_answer\', \'solver_right\',\\n                                \'followed_when_solver_right\', \'solver_wrong\', \'followed_when_solver_wrong\')})\\n    return m, rows\\n\\n\\ndef score_followup(items, base, new, smt, original_metrics=None):\\n    \\"\\"\\"base: {\'initial\': records, \'think\': records} from the first run; new: task -> records.\\"\\"\\"\\n    b = blind(items)\\n    direct, extract = by_item(base[\'initial\'], \'direct\'), by_item(base[\'initial\'], \'extract\')\\n    out, correct = {}, {}\\n\\n    def put(name, rows, extra=None):\\n        m = cls_metrics(rows)\\n        m.update(extra or {})\\n        out[name] = m\\n        correct[name] = [int(r[\'gold\'] == r[\'pred\']) for r in rows]\\n\\n    # reproduction of the first run (same scorer)\\n    rep = score(items, {\'direct\': list(direct.values()), \'think\': base[\'think\'], \'extract\': list(extract.values())},\\n                smt)\\n    out[\'reproduced_first_run\'] = {k: {\'accuracy\': rep[k][\'accuracy\'], \'macro_f1\': rep[k][\'macro_f1\']}\\n                                   for k in (\'direct\', \'think\', \'extract_solve\')}\\n    if original_metrics:\\n        for k, v in out[\'reproduced_first_run\'].items():\\n            assert abs(v[\'accuracy\'] - original_metrics[k][\'accuracy\']) < 1e-12, k\\n    put(\'direct\', [{\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': parse_label(direct[it[\'id\']][\'raw\'])[0]}\\n                   for it in items])\\n\\n    # thinking at three budgets, strict and budget-forced\\n    traces, forced = think_traces(base[\'think\'], new)\\n    budget_table = {}\\n    for B in BUDGETS:\\n        tr = traces[B]\\n        strict = [{\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': parse_think(tr[it[\'id\']][\'raw\'])[0]} for it in items]\\n        put(f\'think{B}_strict\', strict)\\n        unfinished = [it[\'id\'] for it in items if tr[it[\'id\']][\'hit_limit\']]\\n        info = {\'unfinished\': len(unfinished), \'mean_tokens\': sum(tr[i][\'tokens\'] for i in tr) / len(tr)}\\n        if all(i in forced[B] for i in unfinished):\\n            rows = [{\'id\': r[\'id\'], \'gold\': r[\'gold\'],\\n                     \'pred\': parse_think(forced[B][r[\'id\']])[0] if r[\'id\'] in forced[B] else r[\'pred\']}\\n                    for r in strict]\\n            put(f\'think{B}_forced\', rows, {\'forced\': len(unfinished)})\\n            info[\'forced_accuracy\'] = out[f\'think{B}_forced\'][\'accuracy\']\\n        else:\\n            info[\'forced_accuracy\'] = None   # stage not run yet\\n        info[\'strict_accuracy\'] = out[f\'think{B}_strict\'][\'accuracy\']\\n        info[\'strict_parse_failures\'] = out[f\'think{B}_strict\'][\'parse_failures\']\\n        budget_table[B] = info\\n    out[\'think_budget_table\'] = budget_table\\n\\n    # extraction + Z3, before and after one repair round\\n    forms = formalise_all(b, extract, smt)\\n    put(\'extract_solve\', [{\'id\': it[\'id\'], \'gold\': it[\'gold\'], \'pred\': forms[it[\'id\']].get(\'label\', \'parse_failure\')}\\n                          for it in items], {\'extraction\': extraction_quality(items, forms)})\\n    if \'extract_repair\' in new:\\n        rep_recs = by_item(new[\'extract_repair\'])\\n        forms_r = repaired_forms(b, forms, rep_recs, smt)\\n        changed = sum(_norm_claim(forms_r[i].get(\'claim\') or {}) != _norm_claim(forms[i].get(\'claim\') or {})\\n                      or shape_of(forms_r[i][\'inst\'], {}) != shape_of(forms[i][\'inst\'], {})\\n                      for i in rep_recs if \'inst\' in forms_r[i] and \'inst\' in forms[i])\\n        put(\'extract_repair\', [{\'id\': it[\'id\'], \'gold\': it[\'gold\'],\\n                                \'pred\': forms_r[it[\'id\']].get(\'label\', \'parse_failure\')} for it in items],\\n            {\'triggered\': len(rep_recs), \'changed\': changed,\\n             \'triggered_gold\': dict(Counter(it[\'gold\'] for it in items if it[\'id\'] in rep_recs)),\\n             \'extraction\': extraction_quality(items, forms_r)})\\n    else:\\n        forms_r = None\\n\\n    # no LLM in the loop: answer with Z3\'s verdict whenever the extraction is usable\\n    put(\'solver_final\', [{\'id\': it[\'id\'], \'gold\': it[\'gold\'],\\n                          \'pred\': forms[it[\'id\']].get(\'label\') or parse_label(direct[it[\'id\']][\'raw\'])[0]}\\n                         for it in items])\\n\\n    # realistic verify-and-revise\\n    for name, fm in ((\'verify_flag\', forms), (\'verify_evidence\', forms), (\'verify_evidence_repaired\', forms_r)):\\n        if name in new and fm is not None:\\n            m, rows = _verify_condition(items, direct, fm, by_item(new[name]))\\n            out[name] = m\\n            correct[name] = [int(r[\'gold\'] == r[\'pred\']) for r in rows]\\n\\n    comps = {}\\n    pairs = [(\'direct\', \'think16384_forced\'), (\'direct\', \'think16384_strict\'), (\'think3072_strict\', \'think16384_strict\'),\\n             (\'think16384_forced\', \'extract_solve\'), (\'extract_solve\', \'extract_repair\'),\\n             (\'direct\', \'verify_flag\'), (\'direct\', \'verify_evidence\'), (\'direct\', \'verify_evidence_repaired\'),\\n             (\'extract_solve\', \'verify_evidence\'), (\'extract_repair\', \'verify_evidence_repaired\')]\\n    for a, c in pairs:\\n        if a in correct and c in correct:\\n            comps[f\'{c}_minus_{a}\'] = bootstrap_diff(correct[a], correct[c])\\n    out[\'paired_bootstrap\'] = comps\\n    return out\\n\\n\\nMAIN_KEYS = (\'accuracy\', \'macro_f1\', \'premature_certainty\', \'false_uncertainty\', \'missed_inconsistency\',\\n             \'false_inconsistency\', \'parse_failures\')\\n\\n\\ndef main_table(metrics):\\n    return {k: {kk: v[kk] for kk in MAIN_KEYS if kk in v} for k, v in metrics.items()\\n            if isinstance(v, dict) and \'accuracy\' in v}\\n\\n\\n# ---------------------------------------------------------------------------\\n# CLI\\n# ---------------------------------------------------------------------------\\ndef load_base(base_dir):\\n    base_dir = Path(base_dir)\\n    items = read_jsonl(base_dir / \'data/llm/tasks.jsonl\')\\n    llm = base_dir / \'results/llm\'\\n    base = {\'initial\': read_jsonl(llm / \'initial_records.jsonl\'), \'think\': read_jsonl(llm / \'think_records.jsonl\')}\\n    meta = {k: json.loads((llm / f\'{k}_metadata.json\').read_text()) for k in (\'initial\', \'think\')}\\n    original = json.loads((llm / \'metrics.json\').read_text(encoding=\'utf-8\'))\\n    return items, base, meta, original\\n\\n\\ndef load_new(new_dir):\\n    new = {}\\n    for p in sorted(Path(new_dir).glob(\'*_records.jsonl\')):\\n        for r in read_jsonl(p):\\n            new.setdefault(r[\'task\'], []).append(r)\\n    return new\\n\\n\\ndef selftest(base_dir):\\n    \\"\\"\\"Local, no GPU: frozen prompts, blindness, job counts, sample prompts, scorer on fake outputs.\\"\\"\\"\\n    from smt import SMT\\n    smt = SMT()\\n    items, base, meta, original = load_base(base_dir)\\n    assert check_frozen(items, meta[\'initial\'], meta[\'think\'])\\n    b = blind(items)\\n    assert not any(k in it for it in b for k in HIDDEN)\\n    direct, extract = by_item(base[\'initial\'], \'direct\'), by_item(base[\'initial\'], \'extract\')\\n    forms = formalise_all(b, extract, smt)\\n    vf = verify_jobs(b, direct, forms, smt, \'flag\', \'verify_flag\')\\n    ve = verify_jobs(b, direct, forms, smt, \'evidence\', \'verify_evidence\')\\n    rj = repair_jobs(b, extract, forms, smt)\\n    traces = initial_traces(base[\'think\'])\\n    f3 = force_jobs(b, traces, \'think_force3072\')\\n    c8 = continue_jobs(b, traces, 8192, \'think_cont8192\')\\n    print(\'jobs:\', {\'verify_flag\': len(vf), \'verify_evidence\': len(ve), \'extract_repair\': len(rj),\\n                    \'think_force3072\': len(f3), \'think_cont8192\': len(c8)})\\n    print(\'\\\\n--- verify_evidence sample feedback ---\\\\n\' + ve[0][\'messages\'][-1][\'content\'])\\n    print(\'\\\\n--- extract_repair sample feedback ---\\\\n\' + rj[0][\'messages\'][-1][\'content\'])\\n    print(\'\\\\n--- think_force3072 prefix tail ---\\\\n\' + repr(f3[0][\'prefix\'][-160:]))\\n    print(\'budgets:\', {j[\'max_new_tokens\'] for j in c8}, \'batch sizes (95/80/48 GiB, 8k/16k):\',\\n          [(batch_size_for(m, 1800 + 8192 + 64), batch_size_for(m, 1800 + 16384 + 64)) for m in (95, 79, 47)])\\n\\n    # fake GPU outputs to exercise every scoring path\\n    rng = random.Random(0)\\n    fake = lambda jobs, f: [{\'id\': j[\'id\'], \'item\': j[\'item\'], \'task\': j[\'task\'], **f(j)} for j in jobs]\\n    word = [\'成立\', \'不成立\', \'不确定\', \'矛盾\']\\n    new = {\'verify_flag\': fake(vf, lambda j: {\'raw\': rng.choice(word)}),\\n           \'verify_evidence\': fake(ve, lambda j: {\'raw\': rng.choice(word)}),\\n           \'extract_repair\': fake(rj, lambda j: {\'raw\': extract[j[\'item\']][\'raw\']}),\\n           \'think_force3072\': fake(f3, lambda j: {\'raw\': rng.choice(word), \'output_tokens\': 3, \'hit_limit\': False})}\\n    new[\'verify_evidence_repaired\'] = new[\'verify_evidence\']\\n    cont = lambda j: ({\'raw\': \'……\\\\n</think>\\\\n\\\\n\' + rng.choice(word), \'output_tokens\': 900, \'hit_limit\': False}\\n                      if rng.random() < 0.5 else {\'raw\': \'……\', \'output_tokens\': j[\'max_new_tokens\'], \'hit_limit\': True})\\n    new[\'think_cont8192\'] = fake(c8, cont)\\n    tr8 = extend(traces, new[\'think_cont8192\'])\\n    new[\'think_force8192\'] = fake(force_jobs(b, tr8, \'think_force8192\'),\\n                                  lambda j: {\'raw\': rng.choice(word), \'output_tokens\': 3, \'hit_limit\': False})\\n    c16 = continue_jobs(b, tr8, 16384, \'think_cont16384\')\\n    assert {j[\'max_new_tokens\'] for j in c16} == {8192}\\n    new[\'think_cont16384\'] = fake(c16, cont)\\n    tr16 = extend(tr8, new[\'think_cont16384\'])\\n    new[\'think_force16384\'] = fake(force_jobs(b, tr16, \'think_force16384\'),\\n                                   lambda j: {\'raw\': rng.choice(word), \'output_tokens\': 3, \'hit_limit\': False})\\n    m = score_followup(items, base, new, smt, original)\\n    print(\'\\\\nreproduced first run:\', json.dumps(m[\'reproduced_first_run\']))\\n    print(\'think budget table (FAKE outputs):\', json.dumps(m[\'think_budget_table\'], indent=1))\\n    print(\'extract_repair with unchanged JSON (must equal extract_solve):\',\\n          m[\'extract_repair\'][\'accuracy\'], m[\'extract_solve\'][\'accuracy\'], m[\'extract_repair\'][\'changed\'])\\n    assert m[\'extract_repair\'][\'accuracy\'] == m[\'extract_solve\'][\'accuracy\'] and m[\'extract_repair\'][\'changed\'] == 0\\n    assert m[\'think3072_strict\'][\'parse_failures\'] >= original[\'think\'][\'parse_failures\']\\n    print(\'solver_final accuracy:\', m[\'solver_final\'][\'accuracy\'])\\n    print(\'SELFTEST OK\')\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\'--selftest\', action=\'store_true\')\\n    ap.add_argument(\'--score\', action=\'store_true\')\\n    ap.add_argument(\'--base\', default=\'results/colab_run\')\\n    ap.add_argument(\'--new\', default=\'results/followup_run/results_followup/llm\')\\n    ap.add_argument(\'--out\', default=\'results/followup_run/metrics_followup_local.json\')\\n    a = ap.parse_args()\\n    if a.selftest:\\n        selftest(a.base)\\n    if a.score:\\n        from smt import SMT\\n        items, base, _, original = load_base(a.base)\\n        m = score_followup(items, base, load_new(a.new), SMT(), original)\\n        Path(a.out).write_text(json.dumps(m, ensure_ascii=False, indent=2), encoding=\'utf-8\')\\n        print(json.dumps(main_table(m), ensure_ascii=False, indent=1))\\n        print(json.dumps(m[\'think_budget_table\'], indent=1))\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n"}')
for name, content in ASSETS.items():
    p = ROOT / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')
sys.path.insert(0, str(ROOT / 'avalon'))
os.chdir(ROOT)
print('code files:', len(ASSETS))

## 3 输入包：校验、冻结提示词检查、复现第一次运行的评分（不调用模型）

In [ ]:
INPUTS_SHA = '30c050dcf072bfc9d9495a58c5bd5c2a77ba8ba6ad8dbb21bb52cda414580d85'
TASKS_SHA = '256fefe30df81be9bcfde152fe4cf0cb1229b0675a7ad6929b983c7a1dbe59c0'
zp = Path('/content/avalon_followup_inputs.zip')
if not zp.exists():
    from google.colab import files
    up = files.upload()   # 选择 avalon_followup_inputs.zip
    zp.write_bytes(next(iter(up.values())))
assert hashlib.sha256(zp.read_bytes()).hexdigest() == INPUTS_SHA, '输入包校验失败：请上传本次生成的 avalon_followup_inputs.zip'
with zipfile.ZipFile(zp) as z:
    z.extractall(ROOT)
assert hashlib.sha256((ROOT / 'base/data/llm/tasks.jsonl').read_bytes()).hexdigest() == TASKS_SHA
from followup import *
from smt import SMT
smt = SMT()
items, base, meta, original = load_base(ROOT / 'base')
assert check_frozen(items, meta['initial'], meta['think'])
b = blind(items)
direct, extract = by_item(base['initial'], 'direct'), by_item(base['initial'], 'extract')
traces = initial_traces(base['think'])
dry = score_followup(items, base, {}, smt, original)   # asserts the first run's accuracies are reproduced
print(json.dumps(dry['reproduced_first_run'], indent=1))
print('items', len(items), '| thinking traces cut at 3072:', sum(t['hit_limit'] for t in traces.values()),
      '| solver_final accuracy:', round(dry['solver_final']['accuracy'], 4))

## 3b 仅在运行时被回收后使用：从已下载的阶段结果续跑

正常流程直接运行即可（会显示“无需恢复”）。如果运行时被回收，在新运行时里先把最近一次下载的
`avalon_followup_stage*.zip` 上传到 `/content`，再从第 1 节依次运行；已完成的阶段不会重算。

In [ ]:
RES = ROOT / 'results_followup'
_cands = sorted(Path('/content').glob('avalon_followup_stage*.zip'))
if not RES.exists() and _cands:
    with zipfile.ZipFile(_cands[-1]) as z:
        z.extractall(ROOT)
    print('restored from', _cands[-1], sorted(p.name for p in (RES / 'llm').glob('*_records.jsonl')))
else:
    print('无需恢复')

## 4 加载同一 revision 的 Qwen3-8B

In [ ]:
from llm_run import Runner
runner = Runner('results_followup/llm')
_t = runner.tok.apply_chat_template([{'role': 'user', 'content': 'x'}], tokenize=False,
                                    add_generation_prompt=True, enable_thinking=True)
assert _t.endswith('<|im_start|>assistant\n'), repr(_t[-80:])
NEW = {}
def run(jobs, name, **kw):
    NEW[name] = runner.run(jobs, name, **kw)
    print(name, ':', len(jobs), 'jobs done', flush=True)

def pack(tag):
    from google.colab import files
    out = ROOT / 'results_followup'
    m = score_followup(items, base, NEW, smt, original)
    (out / 'metrics_followup.json').write_text(json.dumps(m, ensure_ascii=False, indent=2), encoding='utf-8')
    (out / 'STATUS.json').write_text(json.dumps({'stage': tag, 'inputs_sha256': INPUTS_SHA, 'tasks_sha256': TASKS_SHA,
                                                 'records': {k: len(v) for k, v in NEW.items()}}, indent=2))
    print(json.dumps(main_table(m), ensure_ascii=False, indent=1))
    print(json.dumps(m['think_budget_table'], indent=1))
    archive = Path(f'/content/avalon_followup_{tag}.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file() and not p.name.endswith('_jobs.json'):   # jobs are rebuilt deterministically
                z.write(p, p.relative_to(ROOT))
    print(archive, archive.stat().st_size, 'bytes, sha256', hashlib.sha256(archive.read_bytes()).hexdigest())
    try:
        files.download(str(archive))
    except Exception as e:
        print('自动下载失败，请从左侧文件面板手动下载', archive, e)
print(json.dumps(runner.meta, indent=1))

## 5 真实验证流水线与抽取修复（关闭 thinking，greedy；约 15 分钟）

In [ ]:
forms = formalise_all(b, extract, smt)
run(verify_jobs(b, direct, forms, smt, 'flag', 'verify_flag'), 'verify_flag')
run(verify_jobs(b, direct, forms, smt, 'evidence', 'verify_evidence'), 'verify_evidence')
run(repair_jobs(b, extract, forms, smt), 'extract_repair')
forms_r = repaired_forms(b, forms, by_item(NEW['extract_repair']), smt)
run(verify_jobs(b, direct, forms_r, smt, 'evidence', 'verify_evidence_repaired'), 'verify_evidence_repaired')
pack('stage1')

## 6 thinking：3072 处强制作答，续写到 8192 并强制作答（约 1–2 小时）

In [ ]:
BS8 = batch_size_for(MEM, 1800 + 8192 + 64)
BS16 = batch_size_for(MEM, 1800 + 16384 + 64)
print('batch sizes: 8192 ->', BS8, '| 16384 ->', BS16)
run(force_jobs(b, traces, 'think_force3072'), 'think_force3072', batch_size=BS8)
run(continue_jobs(b, traces, 8192, 'think_cont8192'), 'think_cont8192', batch_size=BS8)
tr8 = extend(traces, NEW['think_cont8192'])
run(force_jobs(b, tr8, 'think_force8192'), 'think_force8192', batch_size=BS8)
print('unfinished at 8192:', sum(t['hit_limit'] for t in tr8.values()))
pack('stage2')

## 7 thinking：仍未结束的续写到 16384 并强制作答（时长取决于剩余条数）

In [ ]:
run(continue_jobs(b, tr8, 16384, 'think_cont16384'), 'think_cont16384', batch_size=BS16)
tr16 = extend(tr8, NEW['think_cont16384'])
run(force_jobs(b, tr16, 'think_force16384'), 'think_force16384', batch_size=BS16)
print('unfinished at 16384:', sum(t['hit_limit'] for t in tr16.values()))

## 8 最终评分与打包（下载后断开并删除运行时）

In [ ]:
pack('final')